# PEH-OUD Microsimulation

Cost-effectiveness of a housing intervention for people experiencing homelessness (PEH) with opioid use disorder (OUD), parameterised to Los Angeles County data and the Measure A housing targets.

**Model.** An individual-level state-transition microsimulation with a monthly cycle and a lifetime horizon. Each person occupies a joint housing state (permanently housed **H**, sheltered **S**, unsheltered **U**) and OUD state (in treatment **YT**, out of treatment **NT**, abstinent **AT**). Mortality combines background, homelessness-related, OUD-related and fatal-overdose hazards. Health care and housing costs (2025 USD) and QALYs are discounted at 3% per year.

**Arms.** The status quo plus a threshold grid that scales the intervention's effect on 1-year housing access, M ∈ {0.2, 0.4, 0.6, 0.8, 1.0}; M = 1 is Measure A as observed. The doubled-housing-cost and higher-quality-of-life sensitivity analyses re-cost and re-weight the same simulated life-courses.

**Uncertainty.** Every reported result comes from the probabilistic sensitivity analysis (PSA): 5,000 parameter draws × 50 replications × 18,928 individuals per arm. Results are PSA means with 95% uncertainty intervals (2.5th–97.5th percentiles across draws); ICERs are mean ΔCost / mean ΔQALY.

## Running the notebook

**Inputs.** Place `CDC_Life_Table.csv`, `Male_AgeMultiplier_QALY.csv`, `Female_AgeMultiplier_QALY.csv`, `Male_Healthcare_Cost.csv` and `Female_Healthcare_Cost.csv` in the data directory.

**Directories.** The project root is `$PEH_OUD_ROOT`; otherwise `$SCRATCH/PEH_OUD` on clusters that define `$SCRATCH` (e.g. Stanford Sherlock); otherwise `./PEH_OUD`. Inputs are read from `<root>/data/` and results written to `<root>/V13/` unless `$PEH_OUD_DATA` or `$PEH_OUD_RESULTS` is set.

**Smoke test.** Set `QUICK_TEST = True` (Section 3) to run the full pipeline at reduced scale in a few minutes.

**Parallel runs.** Draws are independent — each draw's parameters and random numbers derive from its `draw_id` alone — so the PSA can be split across Slurm jobs:

1. Submit one job per draw range (`PSA_DRAW_START`, `PSA_DRAW_END`) with `PEH_OUD_SKIP_ANALYSIS=1`, all sharing one results directory.
2. When every shard has finished, run the notebook once with `PEH_OUD_SKIP_ANALYSIS=0`. It merges the shards without re-simulating and writes all tables and figures.

Progress is checkpointed after every batch of draws, so an interrupted job resumes where it stopped.

| Environment variable | Purpose | Default |
|---|---|---|
| `PEH_OUD_ROOT`, `PEH_OUD_DATA`, `PEH_OUD_RESULTS` | Project, input and output directories | see above |
| `PSA_DRAW_START`, `PSA_DRAW_END` | Half-open draw range for this job | `0`, `n_psa` |
| `PSA_ARMS` | Comma-separated subset of arms to simulate | all arms |
| `PSA_SHARD_TAG` | Checkpoint file suffix | the draw range |
| `PEH_OUD_SKIP_ANALYSIS` | `1` = simulate only; skip analysis and export | `0` |
| `PEH_OUD_WORKERS` | Parallel worker processes | `$SLURM_CPUS_PER_TASK`, else 6 |
| `PEH_OUD_BATCH_SIZE` | Draws per checkpoint batch | `max(20, workers)` |

## 1. Setup

In [ ]:
import os

# One BLAS/OpenMP thread per process: parallelism comes from joblib workers.
# Must be set before NumPy is imported.
for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
             "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = "1"

import datetime
import glob
import hashlib
import json
import time
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.linalg import expm
from scipy.optimize import least_squares, minimize_scalar
from scipy.stats import beta as beta_dist, gamma as gamma_dist, lognorm

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False,
})
warnings.filterwarnings("ignore")

FIGDPI = 600   # resolution of saved figures
MAX_AGE = 120  # upper bound of every age-indexed lookup table

## 2. Paths and checkpointing

Results are stored as append-only JSON-Lines checkpoints, one record per PSA draw. Each checkpoint carries a fingerprint of the model inputs (Section 12), so a resumed run cannot mix results produced under different parameter values.

In [ ]:
MODEL_VERSION = "V13"

_default_root = (os.path.join(os.environ["SCRATCH"], "PEH_OUD")
                 if os.environ.get("SCRATCH") else os.path.abspath("./PEH_OUD"))
PROJECT_ROOT = os.environ.get("PEH_OUD_ROOT", _default_root)
DATA_DIR = os.path.abspath(
    os.environ.get("PEH_OUD_DATA", os.path.join(PROJECT_ROOT, "data")))
RESULTS_DIR = os.path.abspath(
    os.environ.get("PEH_OUD_RESULTS", os.path.join(PROJECT_ROOT, MODEL_VERSION)))

for _sub in ["replications", "tables", "parameters", "figures", "checkpoints"]:
    os.makedirs(os.path.join(RESULTS_DIR, _sub), exist_ok=True)

# Input files are looked up in these directories, in order
INPUT_DIRS = [DATA_DIR, ".", "./data", "./inputs",
              os.path.join(RESULTS_DIR, "inputs")]


def rpath(sub, name):
    """Path to `name` inside the results subfolder `sub`."""
    return os.path.join(RESULTS_DIR, sub, name)


def figpath(name):
    return rpath("figures", name)


def find_input(filename):
    """Return the first existing path to an input file."""
    for d in INPUT_DIRS:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not find '{filename}'. Searched: {INPUT_DIRS}")


# ------------------------------------------------------------
# Checkpoint store
# ------------------------------------------------------------
def _json_default(o):
    """JSON encoder for NumPy scalars and arrays."""
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"not JSON serialisable: {type(o)}")


class StaleCheckpointError(RuntimeError):
    """A checkpoint was written under a different parameterisation."""


class Checkpoint:
    """Append-only JSONL store of records keyed by an integer unit id.

    A sidecar `<name>.meta.json` stores the run fingerprint; resuming under a
    different fingerprint raises StaleCheckpointError.
    """

    def __init__(self, name, id_field="unit_id"):
        self.name = name
        self.id_field = id_field
        self.path = rpath("checkpoints", f"{name}.jsonl")
        self.meta_path = rpath("checkpoints", f"{name}.meta.json")

    def read_meta(self):
        if not os.path.exists(self.meta_path):
            return None
        try:
            with open(self.meta_path) as f:
                return json.load(f)
        except json.JSONDecodeError:
            return None

    def verify_fingerprint(self, fingerprint):
        """Record the fingerprint if new; raise if it differs from the stored one."""
        if fingerprint is None:
            return
        meta = self.read_meta()
        if meta is None:
            with open(self.meta_path, "w") as f:
                json.dump({"fingerprint": fingerprint,
                           "model_version": MODEL_VERSION,
                           "created": datetime.datetime.now().isoformat(
                               timespec="seconds")}, f, indent=2)
            return
        if meta.get("fingerprint") != fingerprint:
            raise StaleCheckpointError(
                f"Checkpoint '{self.name}' was written under a different "
                f"parameterisation.\n"
                f"  stored : {meta.get('fingerprint')} "
                f"(model {meta.get('model_version')}, {meta.get('created')})\n"
                f"  current: {fingerprint} (model {MODEL_VERSION})\n"
                f"Change MODEL_VERSION to start a new results folder, or run "
                f"Checkpoint('{self.name}').reset() to discard the old results.")

    def records(self):
        if not os.path.exists(self.path):
            return []
        out = []
        with open(self.path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    continue   # partial final line from a hard interruption
        return out

    def completed_ids(self):
        return {r[self.id_field] for r in self.records() if self.id_field in r}

    def _ends_with_newline(self):
        if not os.path.exists(self.path) or os.path.getsize(self.path) == 0:
            return True
        with open(self.path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            return f.read(1) == b"\n"

    def append(self, recs):
        with open(self.path, "a") as f:
            # Terminate a line cut off by an earlier interruption, so the next
            # record is not concatenated onto it.
            if not self._ends_with_newline():
                f.write("\n")
            for r in recs:
                f.write(json.dumps(r, default=_json_default) + "\n")
            f.flush()
            os.fsync(f.fileno())

    def to_frame(self, key=None):
        """All records as a DataFrame, de-duplicated on `key` (default: id field)."""
        recs = self.records()
        if not recs:
            return pd.DataFrame()
        df = pd.DataFrame(recs)
        key = key or self.id_field
        key = [k for k in ([key] if isinstance(key, str) else key)
               if k in df.columns]
        if key:
            df = (df.drop_duplicates(subset=key, keep="last")
                    .sort_values(key).reset_index(drop=True))
        return df

    def reset(self):
        for p in (self.path, self.meta_path):
            if os.path.exists(p):
                os.remove(p)
        print(f"Checkpoint '{self.name}' cleared.")


def run_checkpointed(name, unit_ids, worker, batch_size, n_workers=1,
                     desc="units", id_field="unit_id", fingerprint=None):
    """Run `worker(unit_id)` over `unit_ids`, skipping units already checkpointed.

    `worker` returns a dict (or list of dicts) containing `id_field`. Results are
    appended after every batch, so an interruption loses at most one batch.
    """
    ck = Checkpoint(name, id_field=id_field)
    ck.verify_fingerprint(fingerprint)
    done = ck.completed_ids()
    todo = [u for u in unit_ids if u not in done]
    print(f"[{name}] {len(done):,} of {len(unit_ids):,} {desc} already done; "
          f"{len(todo):,} remaining.")
    if not todo:
        return ck.to_frame()

    t0 = time.time()
    for start in range(0, len(todo), batch_size):
        batch = todo[start:start + batch_size]
        if n_workers > 1:
            recs = Parallel(n_jobs=n_workers)(delayed(worker)(u) for u in batch)
        else:
            recs = [worker(u) for u in batch]
        flat = []
        for r in recs:
            flat.extend(r) if isinstance(r, list) else flat.append(r)
        ck.append(flat)

        n_done = start + len(batch)
        elapsed = time.time() - t0
        eta = elapsed / n_done * (len(todo) - n_done)
        print(f"  [{name}] {n_done:,}/{len(todo):,} {desc} | elapsed "
              f"{elapsed / 60:6.1f} min | ETA {eta / 60:6.1f} min", flush=True)

    print(f"[{name}] complete in {(time.time() - t0) / 60:.1f} min.")
    return ck.to_frame()


print(f"Model version : {MODEL_VERSION}")
print(f"Input dirs    : {INPUT_DIRS}")
print(f"Results dir   : {RESULTS_DIR}")

## 3. Run settings

In [ ]:
QUICK_TEST = False   # True: reduced-scale smoke test of the whole pipeline

# ------------------------------------------------------------
# Simulation size
# ------------------------------------------------------------
n_i        = 18_928   # individuals per replication
n_psa      = 5_000    # PSA parameter draws
n_reps_psa = 50       # replications per draw
max_cycles = 2_400    # safety cap on monthly cycles (200 years)

# Replications are simulated as stacked cohorts of PSA_REPS_PER_CHUNK x n_i individuals; the chunk size bounds memory per worker (~40 MB at 10 x 18,928).
PSA_REPS_PER_CHUNK = 10

if QUICK_TEST:
    n_i = 1_000
    n_psa = 20; n_reps_psa = 10; PSA_REPS_PER_CHUNK = 5

# ------------------------------------------------------------
# Discounting: 3% per year, applied monthly
# ------------------------------------------------------------
annual_d_c = 0.03
annual_d_e = 0.03
d_c = (1 + annual_d_c) ** (1/12) - 1
d_e = (1 + annual_d_e) ** (1/12) - 1

# ------------------------------------------------------------
# Seeds. Each draw derives its own seeds from these, so results do not depend
# on how the PSA is split into jobs or batches.
# ------------------------------------------------------------
COHORT_SEED    = 681
PSA_PARAM_SEED = 7_000_000
PSA_SIM_SEED   = 9_000_000

# ------------------------------------------------------------
# States
# ------------------------------------------------------------
v_housing = ["H", "S", "U"]     # permanently housed, sheltered, unsheltered
v_OUD     = ["YT", "NT", "AT"]  # in treatment, out of treatment, abstinent
OUD_idx   = {s: i for i, s in enumerate(v_OUD)}
H_IDX, S_IDX, U_IDX    = 0, 1, 2
YT_IDX, NT_IDX, AT_IDX = 0, 1, 2

# ------------------------------------------------------------
# Willingness-to-pay values for INMB, CEAC, ENMB and EVPI
# ------------------------------------------------------------
WTP_GRID   = np.arange(0, 300_001, 1_000)
WTP_REPORT = [50_000, 100_000, 150_000]

# ------------------------------------------------------------
# Threshold grid over the intervention's housing effect
# ------------------------------------------------------------
# M scales the intervention's incremental effect on both 1-year housing-access
# probabilities: target(M) = target_SQ + M * (target_MeasureA - target_SQ).
# M = 1 reproduces Measure A. Each grid value is simulated as its own arm.
THRESHOLD_GRID      = [0.20, 0.40, 0.60, 0.80, 1.00]
THRESHOLD_MEASURE_A = 1.00

# Optional "higher chance of housing placement" sensitivity analysis: adds an
# arm at M = DSA3_EFFECT_MULTIPLIER. Not part of the reported analysis.
RUN_DSA3_PLACEMENT     = False
DSA3_EFFECT_MULTIPLIER = 2.0

_extra_M = ([DSA3_EFFECT_MULTIPLIER]
            if RUN_DSA3_PLACEMENT and DSA3_EFFECT_MULTIPLIER not in THRESHOLD_GRID
            else [])
THRESHOLD_ARM_M = sorted(THRESHOLD_GRID + _extra_M)

# ------------------------------------------------------------
# Stored outputs
# ------------------------------------------------------------
SAVE_PSA_PARAMS = True   # keep each draw's sampled parameters

# Replication-level outcomes for the empirical convergence check (~60 MB at full
# scale; set REP_TRACK_ARMS = ["sq", "thr1p0"] to store two arms only).
SAVE_PER_REPLICATION = True
CHUNK_TRACK          = ("cost_base", "qaly_base")
REP_TRACK_ARMS       = None   # None = every arm

# ------------------------------------------------------------
# Parallel execution (affects run time only, not results)
# ------------------------------------------------------------
_slurm_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", "0") or 0)
n_workers = max(1, int(os.environ.get("PEH_OUD_WORKERS", _slurm_cpus or 6)))
if _slurm_cpus:
    n_workers = min(n_workers, _slurm_cpus)
PSA_CHECKPOINT_EVERY = 20
PSA_BATCH_SIZE = int(os.environ.get("PEH_OUD_BATCH_SIZE",
                                    max(PSA_CHECKPOINT_EVERY, n_workers)))

# ------------------------------------------------------------
# Sharding (see the overview)
# ------------------------------------------------------------
PSA_DRAW_START = int(os.environ.get("PSA_DRAW_START", 0))
PSA_DRAW_END   = int(os.environ.get("PSA_DRAW_END", n_psa))
PSA_ARM_SUBSET = os.environ.get("PSA_ARMS")
SHARD_TAG      = os.environ.get("PSA_SHARD_TAG",
                                f"{PSA_DRAW_START:06d}_{PSA_DRAW_END:06d}")
SKIP_ANALYSIS  = (os.environ.get("PEH_OUD_SKIP_ANALYSIS", "0").strip().lower()
                  in {"1", "true", "yes", "y"})

print(f"QUICK_TEST     : {QUICK_TEST}")
print(f"PSA            : {n_psa:,} draws x {n_reps_psa} replications x "
      f"{n_i:,} individuals x {1 + len(THRESHOLD_ARM_M)} arms")
print(f"Discounting    : monthly {d_c:.6f} (costs), {d_e:.6f} (QALYs)")
print(f"Threshold arms : M = {THRESHOLD_ARM_M}")
print(f"This job       : draws [{PSA_DRAW_START:,}, {PSA_DRAW_END:,}), "
      f"arms {PSA_ARM_SUBSET or 'all'}, checkpoint psa_{SHARD_TAG}")
print(f"Workers        : {n_workers} (batch {PSA_BATCH_SIZE}; "
      f"Slurm CPUs {_slurm_cpus or 'n/a'})")
print(f"Skip analysis  : {SKIP_ANALYSIS}")

## 4. Helper functions

In [ ]:
# ------------------------------------------------------------
# Rates and probabilities
# ------------------------------------------------------------
def rate_to_prob(r, t=1/12):
    """Annual rate -> probability of at least one event within t years."""
    r = np.asarray(r, dtype=float)
    if np.any(r < 0):
        raise ValueError("Rate must be >= 0.")
    return 1 - np.exp(-r * t)


def prob_to_rate(p, t=1.0):
    """Probability over t years -> annual rate."""
    p = np.asarray(p, dtype=float)
    if np.any((p < 0) | (p >= 1)):
        raise ValueError("Probability must lie in [0, 1).")
    return -(1.0 / t) * np.log(1 - p)


def competing_split(p_dest, t=1.0):
    """Observed destination proportions over t years -> competing annual rates.

    The total exit rate is derived from the total exit proportion and then
    apportioned in proportion to the destination shares.
    """
    p_dest = np.asarray(p_dest, dtype=float)
    p_tot = p_dest.sum()
    if not (0 < p_tot < 1):
        raise ValueError(f"Total exit proportion must lie in (0, 1); got {p_tot}.")
    r_tot = -np.log(1 - p_tot) / t
    return r_tot * p_dest / p_tot


def make_age_lut(age, value):
    """Lookup array indexed by integer age 0..MAX_AGE.

    Ages outside the supplied range take the nearest supplied value; interior
    gaps are forward-filled.
    """
    v = np.full(MAX_AGE + 1, np.nan)
    a = np.clip(np.asarray(age, dtype=int), 0, MAX_AGE)
    v[a] = np.asarray(value, dtype=float)
    known = np.where(~np.isnan(v))[0]
    if known.size == 0:
        raise ValueError("No usable values supplied.")
    v[:known[0]] = v[known[0]]
    v[known[-1] + 1:] = v[known[-1]]
    for i in range(1, v.size):
        if np.isnan(v[i]):
            v[i] = v[i - 1]
    return v


# ------------------------------------------------------------
# Fitting PSA distributions to reported 95% intervals
# ------------------------------------------------------------
# Each fit holds the distribution's mean at the reported point estimate and
# chooses the spread that best matches both bounds on the log scale.
def fit_beta_n(mean, lo, hi):
    """Beta effective sample size n, with a = mean * n and b = (1 - mean) * n."""
    def loss(log_n):
        n = float(np.exp(log_n))
        a, b = beta_dist.ppf([0.025, 0.975], mean * n, (1 - mean) * n)
        return (np.log(a / lo)) ** 2 + (np.log(b / hi)) ** 2

    r = minimize_scalar(loss, bounds=(np.log(2), np.log(1e9)),
                        method="bounded", options={"xatol": 1e-12})
    return float(np.exp(r.x))


def fit_gamma_shape(mean, lo, hi):
    """Gamma shape, with scale = mean / shape."""
    def loss(log_s):
        s = float(np.exp(log_s))
        a, b = gamma_dist.ppf([0.025, 0.975], s, scale=mean / s)
        return (np.log(a / lo)) ** 2 + (np.log(b / hi)) ** 2

    r = minimize_scalar(loss, bounds=(np.log(0.5), np.log(1e6)),
                        method="bounded", options={"xatol": 1e-10})
    return float(np.exp(r.x))


def fit_lognormal_sigma(mean, lo, hi):
    """Lognormal sigma, with mu = ln(mean) - sigma^2 / 2 so that E[X] = mean."""
    def loss(s):
        mu = np.log(mean) - s * s / 2
        a, b = lognorm.ppf([0.025, 0.975], s, scale=np.exp(mu))
        return (np.log(a / lo)) ** 2 + (np.log(b / hi)) ** 2

    r = minimize_scalar(loss, bounds=(1e-4, 3.0), method="bounded",
                        options={"xatol": 1e-12})
    return float(r.x)

## 5. Initial cohort

In [ ]:
_rng_cohort = np.random.default_rng(COHORT_SEED)

# Age (Kushel et al., 2023): age group first, then a uniform integer age within it
age_groups = ["18-24", "25-39", "40-49", "50-64", "65+"]
age_probs  = [0.04, 0.29, 0.24, 0.37, 0.06]
age_ranges = {"18-24": (18, 24), "25-39": (25, 39), "40-49": (40, 49),
              "50-64": (50, 64), "65+": (65, 80)}
_grp = _rng_cohort.choice(len(age_groups), size=n_i, p=age_probs)
init_age = np.array(
    [_rng_cohort.integers(age_ranges[age_groups[g]][0],
                          age_ranges[age_groups[g]][1] + 1) for g in _grp],
    dtype=float)

# Sex: 70% male. 0 = male, 1 = female (row order of every age/sex table)
p_male = 0.70
init_sex_int = (_rng_cohort.random(n_i) >= p_male).astype(int)

# Housing (Assaf et al., 2025): 22.4% sheltered; 77.6% unsheltered
# (56.4% outdoors + 21.2% in vehicles)
p_S = 22.4 / 100
p_U = (56.4 + 21.2) / 100
init_housing_int = np.where(_rng_cohort.random(n_i) < p_S, S_IDX, U_IDX)

# OUD (Assaf et al., 2025): 9.2% in treatment, 90.8% out of treatment
p_YT_init = 9.2 / 100
init_OUD_int = np.where(_rng_cohort.random(n_i) < p_YT_init, YT_IDX, NT_IDX)

print(f"Age     : mean {init_age.mean():.1f} "
      f"(range {init_age.min():.0f}-{init_age.max():.0f})")
print(f"Sex     : M {np.mean(init_sex_int == 0):.1%}, F {np.mean(init_sex_int == 1):.1%}")
print(f"Housing : S {np.mean(init_housing_int == S_IDX):.1%}, "
      f"U {np.mean(init_housing_int == U_IDX):.1%}")
print(f"OUD     : YT {np.mean(init_OUD_int == YT_IDX):.1%}, "
      f"NT {np.mean(init_OUD_int == NT_IDX):.1%}")

## 6. Age- and sex-specific inputs and price year

Each person-month's state utility is multiplied by an age/sex utility multiplier, and each person-month accrues an age/sex-specific background health care cost (2019 USD); both are indexed by `floor(age)`.

All costs are expressed in 2025 USD. Following the Second Panel on Cost-Effectiveness, health care costs use the Personal Health Care (PHC) deflator through the latest historical year and the PCE health index thereafter; housing, a non-health cost, uses the all-items PCE index.

| Input | Base year | Conversion | Factor |
|---|---|---|---|
| Background health care | 2019 | (PHC 116.0 / 103.0) × (PCE-Health 118.896 / 116.048) | 1.15385 |
| Other health care | 2024 | PCE-Health 118.896 / 116.048 | 1.02454 |
| Housing | 2024 | PCE 126.919 / 123.666 | 1.02630 |

In [ ]:
def load_age_sex_csv(male_file, female_file, value_col):
    """Read a male/female pair of age-indexed CSVs into a (2, MAX_AGE+1) array."""
    out = np.zeros((2, MAX_AGE + 1))
    for si, fname in enumerate([male_file, female_file]):
        df = pd.read_csv(find_input(fname))
        df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
        cols = {c.lower(): c for c in df.columns}
        acol, vcol = cols.get("age"), cols.get(value_col.lower())
        if acol is None or vcol is None:
            raise KeyError(f"{fname}: need columns 'Age' and '{value_col}'; "
                           f"found {list(df.columns)}")
        out[si] = make_age_lut(df[acol].astype(int).values,
                               df[vcol].astype(float).values)
    return out


# ------------------------------------------------------------
# Price year and deflators
# ------------------------------------------------------------
COST_YEAR = 2025

PHC_2019,        PHC_2024        = 103.0,   116.0     # PHC expenditure deflator
PCE_HEALTH_2024, PCE_HEALTH_2025 = 116.048, 118.896   # PCE health price index
PCE_2024,        PCE_2025        = 123.666, 126.919   # PCE, all items

DEFL_BG_HEALTH_2019 = (PHC_2024 / PHC_2019) * (PCE_HEALTH_2025 / PCE_HEALTH_2024)
DEFL_HEALTH_2024    = PCE_HEALTH_2025 / PCE_HEALTH_2024
DEFL_HOUSING_2024   = PCE_2025 / PCE_2024

DEFLATORS = {
    "bg_health_2019": (DEFL_BG_HEALTH_2019, "background health care, 2019 USD"),
    "health_2024":    (DEFL_HEALTH_2024,    "health care, 2024 USD"),
    "housing_2024":   (DEFL_HOUSING_2024,   "housing, 2024 USD"),
}


def to_cost_year(value, basis):
    """Convert a cost to COST_YEAR USD with the named deflator."""
    if basis not in DEFLATORS:
        raise KeyError(f"unknown deflator '{basis}'; choose from {list(DEFLATORS)}")
    return np.asarray(value, float) * DEFLATORS[basis][0]


# ------------------------------------------------------------
# Lookup tables: row 0 = male, row 1 = female; column = floor(age)
# ------------------------------------------------------------
qaly_age_mult = load_age_sex_csv("Male_AgeMultiplier_QALY.csv",
                                 "Female_AgeMultiplier_QALY.csv", "Multiplier")
baseline_cost_2019 = load_age_sex_csv("Male_Healthcare_Cost.csv",
                                      "Female_Healthcare_Cost.csv", "Cost")
baseline_cost_lut = to_cost_year(baseline_cost_2019, "bg_health_2019")

print(f"Deflators to {COST_YEAR} USD: background health care (2019) "
      f"{DEFL_BG_HEALTH_2019:.5f}, health care (2024) {DEFL_HEALTH_2024:.5f}, "
      f"housing (2024) {DEFL_HOUSING_2024:.5f}")

_ages = [18, 30, 46, 65, 80]
print(f"\nAge/sex inputs at selected ages ({COST_YEAR} USD)")
print(pd.DataFrame({
    "Age": _ages,
    "Utility mult (M)": [qaly_age_mult[0, a] for a in _ages],
    "Utility mult (F)": [qaly_age_mult[1, a] for a in _ages],
    "Background cost (M)": [baseline_cost_lut[0, a] for a in _ages],
    "Background cost (F)": [baseline_cost_lut[1, a] for a in _ages],
}).to_string(index=False, formatters={
    "Utility mult (M)": "{:.3f}".format, "Utility mult (F)": "{:.3f}".format,
    "Background cost (M)": "${:,.0f}".format,
    "Background cost (F)": "${:,.0f}".format}))

## 7. Mortality and overdose

The monthly probability of death is $1 - \exp(-\lambda/12)$, with annual hazard

$$\lambda = \lambda_\text{BG}(a,s)\,\big[1 + (M_\text{home}(a) - 1)\,\mathbf{1}\{h \in S, U\}\big] + \lambda_\text{OUD}(o) + \lambda_\text{OD}(o)\,p_F(h)$$

- **Background**, $\lambda_\text{BG} = -\ln(1 - q_a)$, from the annual death probability $q_a$ in the `mortality` column of the CDC life table (other columns are ignored).
- **Homelessness**: age-banded mortality multipliers $M_\text{home}$ applied to background mortality while sheltered or unsheltered.
- **OUD non-overdose excess** $\lambda_\text{OUD}$, derived from Ma et al. crude mortality rates as all-cause − overdose − 0.0028 (background at the reference age): 0.0040 (YT), 0.0218 (NT) and 0.0073 (AT, post-cessation proxy) per year, not age-specific. The Ma et al. overdose rates serve only to remove overdose deaths from all-cause mortality, so fatal overdoses are not double-counted.
- **Overdose**: incidence $\lambda_\text{OD}$ depends on treatment status only (0.0543/yr in treatment, 0.1274/yr out of treatment, none while abstinent). Case fatality $p_F$ is one minus the reported per-overdose survival in each housing state: 0.0602 (H), 0.0296 (S), 0.0574 (U).

Overdoses are simulated as events. Each living person-month draws nonfatal overdoses from Poisson($\lambda_\text{OD}(1-p_F)/12$); at each death, the cause is a fatal overdose with probability $\lambda_\text{OD}\,p_F/\lambda$. Total overdoses therefore equal fatal plus nonfatal overdoses in simulated counts.

The homelessness multipliers are published point estimates, but their ±20% PSA range is modeller-assigned; `SAMPLE_HOME_MULT = False` holds them fixed. `PF_UNHOUSED_UNIFORM = True` pools sheltered and unsheltered case fatality as a structural sensitivity analysis.

In [ ]:
# ------------------------------------------------------------
# Background mortality (CDC life table)
# ------------------------------------------------------------
LIFE_TABLE_IS_PROBABILITY = True    # `mortality` holds annual probabilities q_x
LIFE_TABLE_VALUE_COL = "mortality"  # the only mortality column read
_MORT_COLS = ["mortality", "mortality_rate", "rate", "mx", "qx",
              "death_rate", "hazard", "value"]   # fallback column names


def load_life_table():
    """Return (hazards, path, description, ignored columns).

    `hazards` is a (2, MAX_AGE+1) array of annual background mortality hazards.
    Accepted layouts: age + value (both sexes), age + value_male / value_female,
    or long format with a `sex` column.
    """
    path = find_input("CDC_Life_Table.csv")
    df = pd.read_csv(path)
    df.columns = [c.strip().lstrip("\ufeff").lower().replace(" ", "_")
                  for c in df.columns]

    acol = next((c for c in ["age", "age_years", "x"] if c in df.columns), None)
    if acol is None:
        raise KeyError(f"{path}: no age column found; got {list(df.columns)}")
    age = df[acol].astype(float).round().astype(int).values
    vc = LIFE_TABLE_VALUE_COL.lower()

    def _value_col():
        col = vc if vc in df.columns else next(
            (c for c in _MORT_COLS if c in df.columns), None)
        if col is None:
            raise KeyError(f"{path}: no '{vc}' column; got {list(df.columns)}")
        return col

    def _to_hazard(v):
        """Annual probability -> continuous annual hazard, -ln(1 - q)."""
        v = np.asarray(v, dtype=float)
        if not LIFE_TABLE_IS_PROBABILITY:
            return v
        if np.nanmax(v) >= 1.0:
            raise ValueError(f"{path}: '{vc}' contains a value >= 1; set "
                             f"LIFE_TABLE_IS_PROBABILITY = False if it holds hazards.")
        return -np.log(1 - np.clip(v, 0.0, 1 - 1e-12))

    if "sex" in df.columns:
        col = _value_col()
        out = np.zeros((2, MAX_AGE + 1))
        for si, keys in enumerate([("m", "male", "0", "men"),
                                   ("f", "female", "1", "women")]):
            sub = df[df["sex"].astype(str).str.strip().str.lower().isin(keys)]
            if sub.empty:
                raise ValueError(f"{path}: no rows for sex group {keys}")
            out[si] = make_age_lut(
                sub[acol].astype(float).round().astype(int).values,
                _to_hazard(sub[col].values))
        used, src = [acol, "sex", col], f"long format, column '{col}'"
    elif f"{vc}_male" in df.columns and f"{vc}_female" in df.columns:
        male_c, fem_c = f"{vc}_male", f"{vc}_female"
        out = np.vstack([make_age_lut(age, _to_hazard(df[male_c].values)),
                         make_age_lut(age, _to_hazard(df[fem_c].values))])
        used, src = [acol, male_c, fem_c], f"columns '{male_c}' / '{fem_c}'"
    else:
        col = _value_col()
        lut = make_age_lut(age, _to_hazard(df[col].values))
        out = np.vstack([lut, lut])
        used, src = [acol, col], f"column '{col}', both sexes"

    return out, path, src, [c for c in df.columns if c not in used]


bg_mort_lut, _lt_path, _lt_src, _lt_ignored = load_life_table()

# ------------------------------------------------------------
# Homelessness mortality multipliers (sheltered and unsheltered; housed = 1)
# ------------------------------------------------------------
SAMPLE_HOME_MULT = True   # False: hold the multipliers fixed in the PSA

# (age_lo, age_hi, base case, PSA lower, PSA upper); PSA range = +/-20%
HOME_MULT_BANDS = [
    (0,  24,      1.82, 1.456, 2.184),
    (25, 34,      2.49, 1.992, 2.988),
    (35, 54,      1.92, 1.536, 2.304),
    (55, MAX_AGE, 1.63, 1.304, 1.956),
]


def build_home_mult_lut(values=None):
    """Multiplier by floor(age); `values` overrides the base-case band values."""
    vals = [b[2] for b in HOME_MULT_BANDS] if values is None else values
    out = np.ones(MAX_AGE + 1)
    for (lo, hi, *_), m in zip(HOME_MULT_BANDS, vals):
        out[lo:hi + 1] = m
    return out


home_mult_lut = build_home_mult_lut()

# ------------------------------------------------------------
# OUD non-overdose excess mortality (Ma et al.)
# ------------------------------------------------------------
# Excess = all-cause - overdose - background anchor, floored at zero. PC
# (post-cessation) is the proxy for the abstinent state.
BACKGROUND_ANCHOR = 0.0028   # background mortality at the reference age

MA_AC_BASE = {"YT": 0.0092, "NT": 0.0489, "PC": 0.0169}   # all-cause CMR
MA_OD_BASE = {"YT": 0.0024, "NT": 0.0243, "PC": 0.0068}   # overdose CMR


def oud_nonod_excess(ma_ac=None, ma_od=None, background=BACKGROUND_ANCHOR):
    """Annual non-overdose excess rates [YT, NT, AT]."""
    ac = MA_AC_BASE if ma_ac is None else ma_ac
    od = MA_OD_BASE if ma_od is None else ma_od
    return np.array([max(ac[k] - od[k] - background, 0.0)
                     for k in ("YT", "NT", "PC")])


oud_nonod_base = oud_nonod_excess()

# ------------------------------------------------------------
# Overdose incidence and case fatality
# ------------------------------------------------------------
# Annual incidence by treatment status (same in every housing state), as
# (mean, 95% CI low, 95% CI high). None while abstinent.
OD_RATE_REPORTED = {
    "YT": (0.0543, 0.0274, 0.1036),
    "NT": (0.1274, 0.0630, 0.2454),
}
OD_RATE_YT = OD_RATE_REPORTED["YT"][0]
OD_RATE_NT = OD_RATE_REPORTED["NT"][0]
TOTAL_OD_BASE = np.tile([OD_RATE_YT, OD_RATE_NT, 0.0], (3, 1))   # [housing, OUD]

# Per-overdose survival by housing state, (mean, 95% CI low, 95% CI high).
# Case fatality = 1 - survival.
OD_SURVIVAL_REPORTED = {
    "H": (0.9398, 0.9386, 0.9409),
    "S": (0.9704, 0.9689, 0.9719),
    "U": (0.9426, 0.9397, 0.9452),
}
P_FATAL_BY_HOUSING = {k: 1.0 - v[0] for k, v in OD_SURVIVAL_REPORTED.items()}
P_FATAL_HOUSED_BASE      = P_FATAL_BY_HOUSING["H"]   # 0.0602
P_FATAL_SHELTERED_BASE   = P_FATAL_BY_HOUSING["S"]   # 0.0296
P_FATAL_UNSHELTERED_BASE = P_FATAL_BY_HOUSING["U"]   # 0.0574

# PSA: Beta on the fatality scale, with the effective sample size fitted to each
# reported survival interval (survival ~ Beta(a, b) => fatality ~ Beta(b, a)).
PF_EFFECTIVE_N = {k: fit_beta_n(*v) for k, v in OD_SURVIVAL_REPORTED.items()}
PF_BETA = {k: (P_FATAL_BY_HOUSING[k] * PF_EFFECTIVE_N[k],
               OD_SURVIVAL_REPORTED[k][0] * PF_EFFECTIVE_N[k])
           for k in OD_SURVIVAL_REPORTED}

# Structural sensitivity analysis: pool S and U case fatality (unweighted mean)
PF_UNHOUSED_UNIFORM = False


def build_od_matrices(total_od=None, p_fatal=None, uniform_unhoused=None):
    """Return (total, fatal, nonfatal) annual overdose rates [housing, OUD] and
    the case-fatality vector p_fatal ordered H, S, U."""
    T = TOTAL_OD_BASE if total_od is None else np.asarray(total_od, float)
    pf = (np.array([P_FATAL_HOUSED_BASE, P_FATAL_SHELTERED_BASE,
                    P_FATAL_UNSHELTERED_BASE]) if p_fatal is None
          else np.asarray(p_fatal, float).copy())
    uniform = PF_UNHOUSED_UNIFORM if uniform_unhoused is None else uniform_unhoused
    if uniform:
        pf[S_IDX] = pf[U_IDX] = 0.5 * (pf[S_IDX] + pf[U_IDX])
    pf = np.clip(pf, 0.0, 1.0)
    return T, T * pf[:, None], T * (1 - pf[:, None]), pf


total_od_base, fatal_od_base, nonfatal_od_base, p_fatal_base = build_od_matrices()

# ------------------------------------------------------------
# Checks and summary
# ------------------------------------------------------------
assert np.allclose(fatal_od_base + nonfatal_od_base, total_od_base)
for k, want in [("YT", 0.0040), ("NT", 0.0218), ("AT", 0.0073)]:
    got = oud_nonod_base[OUD_idx[k]]
    assert abs(got - want) < 5e-4, f"non-overdose excess {k}: {got} != {want}"

print(f"Life table: {_lt_path} ({_lt_src}; ignored: {', '.join(_lt_ignored) or 'none'})")
if np.allclose(bg_mort_lut[0], bg_mort_lut[1]):
    print("  No sex split in the life table; the same hazards are used for M and F.")
print(f"  Background hazard at ages 40 / 65 / 90 (male): {bg_mort_lut[0, 40]:.5f} / "
      f"{bg_mort_lut[0, 65]:.5f} / {bg_mort_lut[0, 90]:.5f} per year")
print(f"\nOUD non-overdose excess (YT, NT, AT): "
      f"{', '.join(f'{x:.4f}' for x in oud_nonod_base)} per year")
print(f"Homelessness multipliers (sampled in PSA: {SAMPLE_HOME_MULT}):")
print(pd.DataFrame([{"Age band": f"{lo}-{hi if hi < MAX_AGE else '+'}",
                     "Base": b, "PSA lower": l, "PSA upper": u}
                    for lo, hi, b, l, u in HOME_MULT_BANDS]).to_string(index=False))
print("\nOverdose case fatality by housing state:")
print(pd.DataFrame([{
    "Housing": k, "Survival (95% CI)": f"{m:.4f} ({lo:.4f}-{hi:.4f})",
    "Effective n": f"{PF_EFFECTIVE_N[k]:,.0f}", "P(fatal | OD)": f"{p_fatal_base[i]:.4f}"}
    for i, (k, (m, lo, hi)) in enumerate(OD_SURVIVAL_REPORTED.items())
]).to_string(index=False))
print("\nFatal overdose rate [housing x OUD], per year:")
print(pd.DataFrame(fatal_od_base, index=v_housing, columns=v_OUD).round(5))

## 8. Housing transitions

Housing transition rates are derived from Los Angeles County outcomes:

- **S → U and S → H**: observed annual destination shares (3,582 / 44,840 and 5,054 / 23,275), converted to rates with a competing-risks transformation.
- **H → U**: 12-month return to homelessness of 0.134, $r = -\ln(1-p)$. Loss of permanent housing always leads to the unsheltered state.
- **U → S and U → H**: solved jointly so that, starting unsheltered, the continuous-time probabilities of *ever* reaching S and H within one year equal the targets below. Reaching H includes the indirect U → S → H path.

| 1-year target (ever reached) | Status quo | Measure A |
|---|---|---|
| Interim housing (S) | 22,804 / 66,919 = 0.3408 | 0.45 (policy goal) |
| Permanent housing (H) | 5,937 / 66,919 = 0.0887 | 10,687 / 66,919 = 0.1597 |

The raw counts drive the deterministic calibration; separately supplied Beta posteriors are used only for PSA sampling. In each PSA draw the status quo is recalibrated to that draw's sampled probabilities, and each intervention arm to its fixed target, so every draw pursues the same policy goals against a draw-specific counterfactual.

In [ ]:
# ------------------------------------------------------------
# LA County observations (deterministic calibration)
# ------------------------------------------------------------
RAW_INTERIM_N, RAW_INTERIM_DENOM = 22804, 66919   # accessed interim housing
RAW_PERM_N,    RAW_PERM_DENOM    = 5937,  66919   # reached permanent housing
RAW_S_TO_U_N,  RAW_S_TO_U_DENOM  = 3582,  44840   # sheltered -> unsheltered
RAW_S_TO_H_N,  RAW_S_TO_H_DENOM  = 5054,  23275   # sheltered -> permanent
RAW_H_TO_U_P                     = 0.134          # 12-month return to homelessness

# 1-year probability of ever reaching each state, starting unsheltered
TARGET_SQ_EVER_S = RAW_INTERIM_N / RAW_INTERIM_DENOM   # 0.34077
TARGET_SQ_EVER_H = RAW_PERM_N / RAW_PERM_DENOM         # 0.08872

RAW_PERM_N_INT    = 10687                              # Measure A placements
TARGET_INT_EVER_S = 0.45                               # Measure A policy goal
TARGET_INT_EVER_H = RAW_PERM_N_INT / RAW_PERM_DENOM    # 0.15970

TARGET_SQ  = (TARGET_SQ_EVER_S, TARGET_SQ_EVER_H)
TARGET_INT = (TARGET_INT_EVER_S, TARGET_INT_EVER_H)

HOUSING_VALID_MONTHS = 12   # window of the internal housing validation

# ------------------------------------------------------------
# Beta posteriors (PSA sampling only)
# ------------------------------------------------------------
BETA_INTERIM_1YR = (22805.0, 44116.0)
BETA_PERM_1YR    = (5938.0, 60983.0)
BETA_S_TO_U_PROP = (3583.0, 41259.0)
BETA_S_TO_H_PROP = (5055.0, 18222.0)
BETA_H_TO_U_1YR  = (1793.8, 11587.2)

# Reference central rates, used only to check the calibration
R_U_TO_S_SQ_REF, R_U_TO_H_SQ_REF   = 0.4302, 0.05840
R_U_TO_S_INT_REF, R_U_TO_H_INT_REF = 0.6506, 0.1459
R_S_TO_U_REF, R_S_TO_H_REF         = 0.09479, 0.25765
R_H_TO_U_REF                       = 0.14387


def sheltered_rates(p_to_U, p_to_H):
    """Sheltered destination proportions -> annual (r_S->U, r_S->H)."""
    return competing_split([p_to_U, p_to_H])


def housed_exit_rate(p_return_12mo):
    """12-month return-to-homelessness probability -> annual r_H->U."""
    return float(prob_to_rate(p_return_12mo, t=1.0))


def build_housing_trans(r_US, r_UH, r_SU, r_SH, r_HU):
    """Monthly transition probabilities [current housing, current OUD, next housing].

    Housing transitions do not depend on OUD state; the OUD axis matches the
    engine's indexing.
    """
    T = np.zeros((3, 3, 3))

    p_leave = float(rate_to_prob(r_HU))
    row_H = [1 - p_leave, 0.0, p_leave]

    r_tot = r_SH + r_SU
    p_leave = float(rate_to_prob(r_tot))
    row_S = [p_leave * r_SH / r_tot, 1 - p_leave, p_leave * r_SU / r_tot]

    r_tot = r_UH + r_US
    p_leave = float(rate_to_prob(r_tot))
    row_U = [p_leave * r_UH / r_tot, p_leave * r_US / r_tot, 1 - p_leave]

    for o_i in range(3):
        T[H_IDX, o_i, :] = row_H
        T[S_IDX, o_i, :] = row_S
        T[U_IDX, o_i, :] = row_U
    return T


# ------------------------------------------------------------
# 1-year "ever reached" probabilities, starting unsheltered
# ------------------------------------------------------------
# Evaluated on an expanded 12-state chain: (location, visited S, visited H).
def _ever_idx(loc, seen_S, seen_H):
    return loc * 4 + seen_S * 2 + seen_H


def _ever_probs(v):
    p_S = sum(v[_ever_idx(l, 1, b)] for l in range(3) for b in (0, 1))
    p_H = sum(v[_ever_idx(l, a, 1)] for l in range(3) for a in (0, 1))
    return p_S, p_H


def one_year_access_probs(r_US, r_UH, r_SU, r_SH, r_HU, t=1.0):
    """Continuous time: P(ever reached S), P(ever reached H) within t years."""
    Q = np.zeros((12, 12))

    def add(loc, sS, sH, nloc, rate):
        i = _ever_idx(loc, sS, sH)
        j = _ever_idx(nloc, int(sS or nloc == S_IDX), int(sH or nloc == H_IDX))
        Q[i, j] += rate
        Q[i, i] -= rate

    for sS in (0, 1):
        for sH in (0, 1):
            add(H_IDX, sS, sH, U_IDX, r_HU)
            add(S_IDX, sS, sH, H_IDX, r_SH)
            add(S_IDX, sS, sH, U_IDX, r_SU)
            add(U_IDX, sS, sH, H_IDX, r_UH)
            add(U_IDX, sS, sH, S_IDX, r_US)

    v = np.zeros(12)
    v[_ever_idx(U_IDX, 0, 0)] = 1.0
    return _ever_probs(v @ expm(Q * t))


def monthly_access_probs(r_US, r_UH, r_SU, r_SH, r_HU, months=12):
    """Same quantity stepped through the monthly matrix the simulation uses.

    The gap from the continuous-time target is pure time discretisation.
    """
    T = build_housing_trans(r_US, r_UH, r_SU, r_SH, r_HU)[:, 0, :]
    M = np.zeros((12, 12))
    for loc in range(3):
        for sS in (0, 1):
            for sH in (0, 1):
                for nl in range(3):
                    if T[loc, nl] == 0:
                        continue
                    M[_ever_idx(loc, sS, sH),
                      _ever_idx(nl, int(sS or nl == S_IDX),
                                int(sH or nl == H_IDX))] += T[loc, nl]
    v = np.zeros(12)
    v[_ever_idx(U_IDX, 0, 0)] = 1.0
    for _ in range(months):
        v = v @ M
    return _ever_probs(v)


def calibrate_unsheltered_rates(target_interim, target_perm,
                                r_SU, r_SH, r_HU, x0=(0.43, 0.06)):
    """Solve jointly for (r_U->S, r_U->H) that reproduce both 1-year targets.

    Returns (r_U->S, r_U->H, least-squares cost).
    """
    def residual(log_x):
        rus, ruh = np.exp(log_x)
        pS, pH = one_year_access_probs(rus, ruh, r_SU, r_SH, r_HU)
        return [pS - target_interim, pH - target_perm]

    sol = least_squares(residual, np.log(np.asarray(x0, float)),
                        xtol=1e-13, ftol=1e-13, gtol=1e-13)
    rus, ruh = np.exp(sol.x)
    return float(rus), float(ruh), float(sol.cost)


# ------------------------------------------------------------
# Central rates, derived from the raw observations
# ------------------------------------------------------------
R_S_TO_U, R_S_TO_H = sheltered_rates(RAW_S_TO_U_N / RAW_S_TO_U_DENOM,
                                     RAW_S_TO_H_N / RAW_S_TO_H_DENOM)
R_H_TO_U = housed_exit_rate(RAW_H_TO_U_P)
R_U_TO_S_SQ, R_U_TO_H_SQ, _res_sq = calibrate_unsheltered_rates(
    *TARGET_SQ, R_S_TO_U, R_S_TO_H, R_H_TO_U)
R_U_TO_S_INT, R_U_TO_H_INT, _res_int = calibrate_unsheltered_rates(
    *TARGET_INT, R_S_TO_U, R_S_TO_H, R_H_TO_U)

# ------------------------------------------------------------
# Checks and summary
# ------------------------------------------------------------
_qa = pd.DataFrame([
    ("r_U->S status quo", R_U_TO_S_SQ,  R_U_TO_S_SQ_REF),
    ("r_U->H status quo", R_U_TO_H_SQ,  R_U_TO_H_SQ_REF),
    ("r_U->S Measure A",  R_U_TO_S_INT, R_U_TO_S_INT_REF),
    ("r_U->H Measure A",  R_U_TO_H_INT, R_U_TO_H_INT_REF),
    ("r_S->U",            R_S_TO_U,     R_S_TO_U_REF),
    ("r_S->H",            R_S_TO_H,     R_S_TO_H_REF),
    ("r_H->U",            R_H_TO_U,     R_H_TO_U_REF),
], columns=["Rate", "Derived", "Reference"])
_qa["Rel. diff"] = (_qa["Derived"] / _qa["Reference"] - 1).map("{:+.3%}".format)

_bm = lambda ab: ab[0] / (ab[0] + ab[1])
_beta_check = pd.DataFrame([
    ("1-yr interim access",  TARGET_SQ_EVER_S, _bm(BETA_INTERIM_1YR)),
    ("1-yr permanent access", TARGET_SQ_EVER_H, _bm(BETA_PERM_1YR)),
    ("S->U annual share", RAW_S_TO_U_N / RAW_S_TO_U_DENOM, _bm(BETA_S_TO_U_PROP)),
    ("S->H annual share", RAW_S_TO_H_N / RAW_S_TO_H_DENOM, _bm(BETA_S_TO_H_PROP)),
    ("12-mo return to homelessness", RAW_H_TO_U_P, _bm(BETA_H_TO_U_1YR)),
], columns=["Quantity", "Raw observation", "Beta mean (PSA)"])

for _, _r in _qa.iterrows():
    assert abs(_r["Derived"] / _r["Reference"] - 1) < 0.01, \
        f"{_r['Rate']}: derived {_r['Derived']:.5f} vs reference {_r['Reference']}"
assert max(_res_sq, _res_int) < 1e-12, "housing calibration did not converge"

print(f"Targets: status quo S {TARGET_SQ_EVER_S:.5f}, H {TARGET_SQ_EVER_H:.5f} | "
      f"Measure A S {TARGET_INT_EVER_S:.5f}, H {TARGET_INT_EVER_H:.5f}")
print("\nDerived annual rates vs reference values")
print(_qa.to_string(index=False, formatters={"Derived": "{:.5f}".format,
                                             "Reference": "{:.5f}".format}))
print("\nRaw observations (calibration) vs Beta means (PSA)")
print(_beta_check.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

## 9. OUD transitions

Annual transition rates differ by housing state. Relapse from abstinence declines linearly from its first-month value to its year-10 value over the current abstinence spell; abstinent individuals cannot move directly into treatment.

In [ ]:
# Annual rates, ordered H, S, U
R_NT_TO_YT = np.array([0.426,  0.221,  0.211])    # out of treatment -> in treatment
R_NT_TO_AT = np.array([0.0791, 0.0074, 0.0074])   # out of treatment -> abstinent
R_YT_TO_NT = np.array([1.125,  1.531,  1.531])    # in treatment -> out of treatment
R_YT_TO_AT = np.array([0.316,  0.029,  0.029])    # in treatment -> abstinent

# Relapse (abstinent -> out of treatment): first month of a spell and year >= 10
R_AT_TO_NT_FIRST = np.array([0.344, 0.462, 0.462])
R_AT_TO_NT_Y10   = np.array([0.023, 0.031, 0.031])
at_relapse_base  = np.column_stack([R_AT_TO_NT_FIRST, R_AT_TO_NT_Y10])   # (3, 2)

AT_INTERP_MONTHS = 119.0   # month 1 -> month 120 (start of year 10)


def interp_AT(vals2, h_idx, spell_months):
    """Interpolate a spell-dependent abstinence quantity.

    `vals2` is (3, 2): [housing, (first month, year >= 10)]. A spell of 0 months
    gives the first-month value; >= AT_INTERP_MONTHS gives the year-10 value.
    """
    frac = np.clip(np.asarray(spell_months, dtype=float) / AT_INTERP_MONTHS, 0.0, 1.0)
    v0 = vals2[h_idx, 0]
    v1 = vals2[h_idx, 1]
    return v0 + frac * (v1 - v0)


def build_OUD_trans(psa_rates=None):
    """Monthly transition probabilities [current OUD, current housing, next OUD].

    The abstinent row uses the first-month relapse rate as a placeholder; the
    engine replaces it every cycle with the spell-dependent rate.
    """
    T = np.zeros((3, 3, 3))
    for h_i, h in enumerate(v_housing):
        if psa_rates is not None:
            r_YN = psa_rates[f"r_YT_to_NT_{h}"]; r_YA = psa_rates[f"r_YT_to_AT_{h}"]
            r_NY = psa_rates[f"r_NT_to_YT_{h}"]; r_NA = psa_rates[f"r_NT_to_AT_{h}"]
            r_AN = psa_rates[f"r_AT_to_NT_first_{h}"]
        else:
            r_YN = R_YT_TO_NT[h_i]; r_YA = R_YT_TO_AT[h_i]
            r_NY = R_NT_TO_YT[h_i]; r_NA = R_NT_TO_AT[h_i]
            r_AN = R_AT_TO_NT_FIRST[h_i]

        r_tot = r_YN + r_YA
        p = float(rate_to_prob(r_tot))
        T[YT_IDX, h_i, :] = [1 - p, p * r_YN / r_tot, p * r_YA / r_tot]

        r_tot = r_NY + r_NA
        p = float(rate_to_prob(r_tot))
        T[NT_IDX, h_i, :] = [p * r_NY / r_tot, 1 - p, p * r_NA / r_tot]

        p = float(rate_to_prob(r_AN))
        T[AT_IDX, h_i, :] = [0.0, p, 1 - p]
    return T


def at_relapse_from_psa(p):
    """(3, 2) relapse-rate table from a PSA parameter draw."""
    return np.column_stack([
        [p[f"r_AT_to_NT_first_{h}"] for h in v_housing],
        [p[f"r_AT_to_NT_y10_{h}"] for h in v_housing],
    ])


OUD_trans_base = build_OUD_trans()
assert np.allclose(OUD_trans_base.sum(axis=2), 1.0), "OUD rows must sum to 1"

print(f"Mean treatment episode, housed: "
      f"{1 / rate_to_prob(R_YT_TO_NT[0] + R_YT_TO_AT[0]):.1f} months")
print(f"Relapse from abstinence, housed: {R_AT_TO_NT_FIRST[0]:.3f}/yr in month 1 "
      f"-> {R_AT_TO_NT_Y10[0]:.3f}/yr from year 10")

## 10. Costs and utilities

Source costs are stored as published, with their base year and category, and converted to 2025 USD in one place. PSA Gamma distributions are fitted to each published 95% CI; housing costs have no published interval and use an assumed 20% coefficient of variation. Abstinence carries no OUD-attributable excess cost, but background and homelessness-related costs still accrue.

Sheltered and unsheltered utilities scale the housed values by multipliers derived from the relative prevalence of complex behavioural health needs, normalised so the prevalence-weighted PEH multiplier equals 0.43 (0.60 in the higher-quality-of-life sensitivity analysis).

In [ ]:
# ------------------------------------------------------------
# Source costs: (mean, 95% CI low, 95% CI high, deflator); CI = None if not published
# ------------------------------------------------------------
HOUSING_COST_CV = 0.20   # assumed CV for costs without a published interval

RAW_COSTS = {
    "c_health_YT":            (7052.0,   3675.0, 10658.0, "health_2024"),
    "c_health_NT":            (8827.0,   7966.0,  9731.0, "health_2024"),
    "c_health_S":             (7117.0,   6423.0,  7855.0, "health_2024"),
    "c_health_U":             (15759.0, 14222.0, 17394.0, "health_2024"),
    "c_treatment_YT":         (8485.0,   7658.0,  9353.0, "health_2024"),
    "c_per_OD_H":             (3084.0,   1321.0,  6152.0, "health_2024"),
    "c_per_OD_S":             (3084.0,   1321.0,  6152.0, "health_2024"),
    "c_per_OD_U":             (3029.0,   1297.0,  6043.0, "health_2024"),
    "c_housing_H":            (24624.0,    None,    None, "housing_2024"),
    "c_emergency_shelter":    (18283.0,    None,    None, "housing_2024"),
    "c_transitional_shelter": (35408.0,    None,    None, "housing_2024"),
}

COST_LABELS = {
    "c_health_YT":            "Excess health care cost, OUD in treatment",
    "c_health_NT":            "Excess health care cost, OUD out of treatment",
    "c_health_S":             "Excess health care cost, temporarily sheltered",
    "c_health_U":             "Excess health care cost, unsheltered",
    "c_treatment_YT":         "Annual OUD treatment cost",
    "c_per_OD_H":             "Health care cost per overdose, formal housing",
    "c_per_OD_S":             "Health care cost per overdose, sheltered",
    "c_per_OD_U":             "Health care cost per overdose, unsheltered",
    "c_housing_H":            "Annual cost of permanent housing",
    "c_emergency_shelter":    "Annual cost of emergency shelter",
    "c_transitional_shelter": "Annual cost of transitional shelter",
}

# Converted values, also exposed as variables (c_health_YT, c_housing_H, ...)
COSTS = {k: float(to_cost_year(v[0], v[3])) for k, v in RAW_COSTS.items()}
globals().update(COSTS)

# PSA Gamma (shape, scale) with the mean fixed at the converted value
COST_CI_SOURCE, COST_PSA = {}, {}
for _k, (_m, _lo, _hi, _b) in RAW_COSTS.items():
    if _lo is not None and _hi is not None:
        _shape = fit_gamma_shape(_m, _lo, _hi)
        COST_CI_SOURCE[_k] = "published"
    else:
        _shape = 1.0 / HOUSING_COST_CV ** 2
        COST_CI_SOURCE[_k] = f"derived, CV = {HOUSING_COST_CV:.0%}"
    COST_PSA[_k] = (_shape, COSTS[_k] / _shape)
COST_PSA_CI = {k: tuple(gamma_dist.ppf([0.025, 0.975], s, scale=sc))
               for k, (s, sc) in COST_PSA.items()}

c_health_AT = 0.0   # no OUD-attributable excess cost while abstinent

# Shelter cost: weighted mix of emergency and transitional shelter
p_emergency_shelter    = 0.15 / p_S
p_transitional_shelter = 1 - p_emergency_shelter
c_shelter_S = (p_emergency_shelter * c_emergency_shelter
               + p_transitional_shelter * c_transitional_shelter)

DSA1_HOUSING_MULT = 2.0   # sensitivity analysis: doubled permanent housing cost

# ------------------------------------------------------------
# Conversion audit table (exported as Table0_cost_conversion.csv)
# ------------------------------------------------------------
_rows = []
for k, (m, lo, hi, basis) in RAW_COSTS.items():
    f = DEFLATORS[basis][0]
    _rows.append({
        "Parameter": COST_LABELS[k],
        "Variable": k,
        "Base year": 2019 if basis == "bg_health_2019" else 2024,
        "Category": "Housing" if basis == "housing_2024" else "Health care",
        "Original ($)": m,
        "Original CI": f"{lo:,.0f}-{hi:,.0f}" if lo is not None else "not reported",
        "CI source": COST_CI_SOURCE[k],
        "PSA gamma shape": COST_PSA[k][0],
        "PSA gamma scale": COST_PSA[k][1],
        "PSA implied CV": 1.0 / np.sqrt(COST_PSA[k][0]),
        f"PSA {COST_YEAR} CI low": COST_PSA_CI[k][0],
        f"PSA {COST_YEAR} CI high": COST_PSA_CI[k][1],
        "Deflator": f,
        f"{COST_YEAR} ($)": COSTS[k],
        f"{COST_YEAR} CI": (f"{lo * f:,.0f}-{hi * f:,.0f}" if lo is not None
                            else "not reported"),
    })

# Background cost enters as an age/sex table; summarised by the cohort mean at entry
_entry_age = np.clip(np.floor(init_age).astype(int), 0, MAX_AGE)
_bg19 = baseline_cost_2019[init_sex_int, _entry_age].mean()
_rows.append({
    "Parameter": "Annual background health care cost (cohort mean at entry)",
    "Variable": "baseline_cost_lut", "Base year": 2019, "Category": "Health care",
    "Original ($)": _bg19, "Original CI": "age/sex table",
    "Deflator": DEFL_BG_HEALTH_2019,
    f"{COST_YEAR} ($)": _bg19 * DEFL_BG_HEALTH_2019,
    f"{COST_YEAR} CI": "age/sex table",
})
_rows.append({
    "Parameter": "Weighted shelter cost (derived)",
    "Variable": "c_shelter_S", "Base year": 2024, "Category": "Housing",
    "Original ($)": c_shelter_S / DEFL_HOUSING_2024, "Original CI": "derived",
    "Deflator": DEFL_HOUSING_2024,
    f"{COST_YEAR} ($)": c_shelter_S, f"{COST_YEAR} CI": "derived",
})
table_cost_conversion = pd.DataFrame(_rows)


# ------------------------------------------------------------
# Cost matrices [housing, OUD], annual
# ------------------------------------------------------------
def build_health_cost_matrix(total_od_mat, c_od=None, c_YT=None, c_NT=None,
                             c_AT=None, c_S=None, c_U=None, c_tx=None):
    """State health care cost, excluding the age/sex background cost.

    Overdose care = cost per event x total (fatal + nonfatal) overdose rate.
    """
    c_od = (c_per_OD_H, c_per_OD_S, c_per_OD_U) if c_od is None else c_od
    c_YT = c_health_YT if c_YT is None else c_YT
    c_NT = c_health_NT if c_NT is None else c_NT
    c_S = c_health_S if c_S is None else c_S
    c_U = c_health_U if c_U is None else c_U
    c_tx = c_treatment_YT if c_tx is None else c_tx
    c_AT = 0.0 if c_AT is None else c_AT
    od = np.asarray(total_od_mat, float) * np.asarray(c_od, float)[:, None]
    return np.array([
        [c_YT + c_tx + od[H_IDX, YT_IDX], c_NT + od[H_IDX, NT_IDX], c_AT],
        [c_YT + c_tx + c_S + od[S_IDX, YT_IDX],
         c_NT + c_S + od[S_IDX, NT_IDX], c_AT + c_S],
        [c_YT + c_tx + c_U + od[U_IDX, YT_IDX],
         c_NT + c_U + od[U_IDX, NT_IDX], c_AT + c_U],
    ])


def build_housing_cost_matrix(c_H=None, c_shelt=None):
    """Annual housing cost; zero while unsheltered."""
    c_H = c_housing_H if c_H is None else c_H
    c_shelt = c_shelter_S if c_shelt is None else c_shelt
    return np.array([[c_H, c_H, c_H],
                     [c_shelt, c_shelt, c_shelt],
                     [0.0, 0.0, 0.0]])


health_cost_matrix = build_health_cost_matrix(total_od_base)
housing_cost_mats_base = {
    "base": build_housing_cost_matrix(),
    "dsa1": build_housing_cost_matrix(c_H=c_housing_H * DSA1_HOUSING_MULT),
}

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
# Relative prevalence of complex behavioural health needs, unsheltered vs sheltered
p_U_outdoor = 56.4 / (56.4 + 21.2)
p_U_vehicle = 1 - p_U_outdoor
p_U_sick = p_U_outdoor * 0.6 + p_U_vehicle * 0.43
p_S_sick = 0.25
U_S_sick_ratio = p_U_sick / p_S_sick

q_YT, q_NT, q_AT_FIRST, q_AT_Y10 = 0.721, 0.668, 0.747, 0.983   # housed
Q_PEH_BASE, Q_PEH_DSA2 = 0.43, 0.60   # prevalence-weighted PEH multiplier


def build_qaly_params(q_PEH=Q_PEH_BASE, qYT=q_YT, qNT=q_NT,
                      qAT1=q_AT_FIRST, qAT10=q_AT_Y10, utilities=None):
    """Return (utility matrix [housing, OUD], abstinent utility [housing, (first, y10)]).

    The abstinent column of the matrix holds the first-month value; the engine
    interpolates over the spell. `utilities` supplies PSA-sampled state
    utilities directly, bypassing the multiplier derivation.
    """
    if utilities is not None:
        qm = np.array([[utilities[f"q_{h}_{o}"] for o in ["YT", "NT", "AT_first"]]
                       for h in v_housing])
        atu = np.array([[utilities[f"q_{h}_AT_first"], utilities[f"q_{h}_AT_y10"]]
                        for h in v_housing])
        return qm, atu

    q_U = q_PEH / (U_S_sick_ratio * p_S + p_U)
    q_S = U_S_sick_ratio * q_U
    qm = np.array([[qYT, qNT, qAT1],
                   [qYT * q_S, qNT * q_S, qAT1 * q_S],
                   [qYT * q_U, qNT * q_U, qAT1 * q_U]])
    atu = np.array([[qAT1, qAT10],
                    [qAT1 * q_S, qAT10 * q_S],
                    [qAT1 * q_U, qAT10 * q_U]])
    return qm, atu


qaly_matrix_base, at_utility_base = build_qaly_params(Q_PEH_BASE)
qaly_matrix_dsa2, at_utility_dsa2 = build_qaly_params(Q_PEH_DSA2)

# ------------------------------------------------------------
# Checks and summary
# ------------------------------------------------------------
for (_h, _o), _v in {(0, 0): 0.721, (1, 0): 0.540, (2, 0): 0.244,
                     (0, 1): 0.668, (1, 1): 0.500, (2, 1): 0.226}.items():
    assert abs(qaly_matrix_base[_h, _o] - _v) < 5e-4, f"utility mismatch {(_h, _o)}"
for _h, (_f, _y) in enumerate([(0.747, 0.983), (0.559, 0.736), (0.253, 0.332)]):
    assert abs(at_utility_base[_h, 0] - _f) < 5e-4, f"abstinent month 1, housing {_h}"
    assert abs(at_utility_base[_h, 1] - _y) < 5e-4, f"abstinent year 10, housing {_h}"

print(f"Costs converted to {COST_YEAR} USD")
print(table_cost_conversion[[
    "Parameter", "Original ($)", "Original CI", "Deflator",
    f"{COST_YEAR} ($)", f"{COST_YEAR} CI", "CI source"]].to_string(
    index=False, formatters={"Original ($)": "{:,.0f}".format,
                             f"{COST_YEAR} ($)": "{:,.0f}".format,
                             "Deflator": "{:.5f}".format}))
print(f"\nShelter mix: {p_emergency_shelter:.1%} emergency, "
      f"{p_transitional_shelter:.1%} transitional")
print(f"\nState health care cost [housing x OUD], {COST_YEAR} USD/yr, "
      f"excluding background cost")
print(pd.DataFrame(health_cost_matrix.round(0), index=v_housing, columns=v_OUD))
print("\nUtility [housing x OUD] (abstinent = month 1)")
print(pd.DataFrame(qaly_matrix_base.round(3), index=v_housing, columns=v_OUD))

## 11. Microsimulation engine

In each monthly cycle, living individuals age one month, face competing mortality from the states they occupy, draw nonfatal overdoses, and then transition housing and OUD states. Costs and QALYs accrue from the entry month with age/sex-specific background costs and utility multipliers. The current abstinence spell is tracked so that relapse risk and abstinent utility change with its duration.

- **Several valuations per pass.** The cost and utility sensitivity analyses do not affect transitions or mortality, so each trajectory is costed under every entry of `housing_cost_mats` (`base`, `dsa1`) and valued under every entry of `qaly_mats` (`base`, `dsa2`) in the same pass.
- **Stacked replications.** Replications share the initial cohort and differ only in their random numbers, so they are simulated as one vectorised cohort of `reps × n_i` individuals (in chunks, to bound memory) and reshaped to recover replication-level means.
- **Common random numbers.** Chunk seeds depend only on the draw, so all arms of a draw use the same random-number streams.

In [ ]:
def vectorized_sample(probs_mat, rng):
    """Draw one category per row of `probs_mat` by inverse CDF."""
    u = rng.random(probs_mat.shape[0])
    return np.argmax(np.cumsum(probs_mat, axis=1) >= u[:, None], axis=1)


def run_microsim(init_age, init_sex, init_housing, init_OUD,
                 d_c, d_e,
                 health_cost_mat, housing_cost_mats,
                 qaly_mats, at_util_mats,
                 baseline_cost_lut, qaly_mult_lut,
                 bg_mort_lut, home_mult_lut, oud_nonod,
                 fatal_od_mat, nonfatal_od_mat,
                 housing_trans, OUD_trans, at_relapse,
                 seed=1, max_cycles=2400):
    """Simulate a cohort to a lifetime horizon; return per-individual outcomes.

    `housing_cost_mats`, `qaly_mats` and `at_util_mats` are dicts of
    variant name -> matrix; costs and QALYs are returned for every variant.
    """
    rng = np.random.default_rng(seed)
    n = init_age.size

    age = np.asarray(init_age, float).copy()
    sex = np.asarray(init_sex, int).copy()
    h_int = np.asarray(init_housing, int).copy()
    o_int = np.asarray(init_OUD, int).copy()
    alive = np.ones(n, dtype=bool)
    spell_AT = np.zeros(n)   # months in the current abstinence spell

    cost_names = list(housing_cost_mats)
    qaly_names = list(qaly_mats)
    health_cost = np.zeros(n)
    housing_cost = {k: np.zeros(n) for k in cost_names}
    qaly = {k: np.zeros(n) for k in qaly_names}

    age_at_death = np.zeros(n)
    cycle_of_death = np.full(n, -1, dtype=int)
    deaths = np.zeros(n)
    months = {k: np.zeros(n) for k in ["H", "S", "U", "YT", "AT"]}

    # Overdose events: fatal is a 0/1 indicator (also the cause of death),
    # nonfatal a Poisson count; both are also split by housing state at the event.
    od_fatal = np.zeros(n)
    od_nonfatal = np.zeros(n)
    od_fatal_by_h = [np.zeros(n) for _ in range(3)]
    od_nonfatal_by_h = [np.zeros(n) for _ in range(3)]

    # Internal housing validation: among those starting unsheltered, who ever
    # reached S / H within HOUSING_VALID_MONTHS, and who survived that window
    started_U = (h_int == U_IDX)
    ever_S_12mo = np.zeros(n, dtype=bool)
    ever_H_12mo = np.zeros(n, dtype=bool)
    alive_12mo = np.zeros(n, dtype=bool)

    def _accrue(idx, h, o, spell, df_c, df_e):
        a_i = np.clip(np.floor(age[idx]).astype(int), 0, MAX_AGE)
        s_i = sex[idx]
        health_cost[idx] += (baseline_cost_lut[s_i, a_i]
                             + health_cost_mat[h, o]) / 12.0 * df_c
        for k in cost_names:
            housing_cost[k][idx] += housing_cost_mats[k][h, o] / 12.0 * df_c
        age_mult = qaly_mult_lut[s_i, a_i]
        is_at = (o == AT_IDX)
        for k in qaly_names:
            u = qaly_mats[k][h, o].astype(float)
            if np.any(is_at):
                u = u.copy()
                u[is_at] = interp_AT(at_util_mats[k], h[is_at], spell[is_at])
            qaly[k][idx] += u * age_mult / 12.0 * df_e

    def _count_states(idx, h, o):
        for key, val in [("H", H_IDX), ("S", S_IDX), ("U", U_IDX)]:
            months[key][idx] += (h == val)
        months["YT"][idx] += (o == YT_IDX)
        months["AT"][idx] += (o == AT_IDX)

    # Entry month: costs and QALYs accrue; overdoses are drawn from cycle 1 so
    # that overdose and mortality exposure cover the same person-months.
    all_idx = np.arange(n)
    _accrue(all_idx, h_int, o_int, spell_AT, 1.0, 1.0)
    _count_states(all_idx, h_int, o_int)

    last_cycle = 0
    for t in range(1, max_cycles + 1):
        idx = np.where(alive)[0]
        if idx.size == 0:
            break
        last_cycle = t
        df_c = 1.0 / (1 + d_c) ** t
        df_e = 1.0 / (1 + d_e) ** t

        age[idx] += 1 / 12
        a_i = np.clip(np.floor(age[idx]).astype(int), 0, MAX_AGE)
        s_i = sex[idx]
        cur_h = h_int[idx]
        cur_o = o_int[idx]

        # ---- Competing mortality ----
        bg = bg_mort_lut[s_i, a_i]
        home_excess = np.where(cur_h == H_IDX, 0.0, (home_mult_lut[a_i] - 1.0) * bg)
        fatal_rate = fatal_od_mat[cur_h, cur_o]
        hazard = np.maximum(bg + home_excess + oud_nonod[cur_o] + fatal_rate, 1e-12)

        died_mask = rng.random(idx.size) < 1 - np.exp(-hazard / 12.0)
        died = idx[died_mask]
        survived = idx[~died_mask]

        if died.size:
            # Cause of death drawn from the relative cause-specific hazard
            is_od_death = rng.random(died.size) < (fatal_rate[died_mask]
                                                   / hazard[died_mask])
            od_died = died[is_od_death]
            if od_died.size:
                od_fatal[od_died] = 1.0
                h_at_death = cur_h[died_mask][is_od_death]
                for hv in range(3):
                    m = h_at_death == hv
                    if m.any():
                        od_fatal_by_h[hv][od_died[m]] = 1.0
            alive[died] = False
            deaths[died] = 1.0
            age_at_death[died] = age[died]
            cycle_of_death[died] = t

        # ---- Nonfatal overdoses over the month just lived ----
        counts = rng.poisson(nonfatal_od_mat[cur_h, cur_o] / 12.0)
        od_nonfatal[idx] += counts
        for hv in range(3):
            m = cur_h == hv
            if m.any():
                od_nonfatal_by_h[hv][idx[m]] += counts[m]

        # ---- State transitions for survivors ----
        if survived.size:
            s_h = h_int[survived]
            s_o = o_int[survived]
            s_spell = spell_AT[survived]

            new_h = vectorized_sample(housing_trans[s_h, s_o, :], rng)

            probs = OUD_trans[s_o, s_h, :].astype(float)
            is_at = (s_o == AT_IDX)
            if np.any(is_at):
                probs = probs.copy()
                r_rel = interp_AT(at_relapse, s_h[is_at], s_spell[is_at])
                p_rel = 1.0 - np.exp(-r_rel / 12.0)
                probs[is_at, YT_IDX] = 0.0
                probs[is_at, NT_IDX] = p_rel
                probs[is_at, AT_IDX] = 1.0 - p_rel
            new_o = vectorized_sample(probs, rng)

            still_at = (new_o == AT_IDX) & (s_o == AT_IDX)
            new_spell = np.zeros(survived.size)
            new_spell[still_at] = s_spell[still_at] + 1
            spell_AT[survived] = new_spell
            h_int[survived] = new_h
            o_int[survived] = new_o

            _accrue(survived, new_h, new_o, new_spell, df_c, df_e)
            _count_states(survived, new_h, new_o)

            if t <= HOUSING_VALID_MONTHS:
                sv_U = started_U[survived]
                if sv_U.any():
                    ever_S_12mo[survived[sv_U & (new_h == S_IDX)]] = True
                    ever_H_12mo[survived[sv_U & (new_h == H_IDX)]] = True

        if t == HOUSING_VALID_MONTHS:
            alive_12mo[alive] = True

    result = {
        "health_cost": health_cost,
        "age_at_death": age_at_death,
        "cycle_of_death": cycle_of_death,
        "deaths": deaths,
        "od_fatal": od_fatal,
        "od_nonfatal": od_nonfatal,
        "od_total": od_fatal + od_nonfatal,
        "last_cycle": last_cycle,
        # Unconditional indicators: dividing replication means by the
        # started_U share recovers the conditional probabilities.
        "acc_started_U": started_U.astype(float),
        "acc_ever_S": (started_U & ever_S_12mo).astype(float),
        "acc_ever_H": (started_U & ever_H_12mo).astype(float),
        "acc_alive_12mo": (started_U & alive_12mo).astype(float),
        "acc_ever_S_alive": (started_U & ever_S_12mo & alive_12mo).astype(float),
        "acc_ever_H_alive": (started_U & ever_H_12mo & alive_12mo).astype(float),
    }
    for hv, hlab in enumerate(v_housing):
        result[f"od_fatal_{hlab}"] = od_fatal_by_h[hv]
        result[f"od_nonfatal_{hlab}"] = od_nonfatal_by_h[hv]
        result[f"od_total_{hlab}"] = od_fatal_by_h[hv] + od_nonfatal_by_h[hv]
    for k in cost_names:
        result[f"housing_cost_{k}"] = housing_cost[k]
        result[f"cost_{k}"] = health_cost + housing_cost[k]
    for k in qaly_names:
        result[f"qaly_{k}"] = qaly[k]
    for k, v in months.items():
        result[f"months_{k}"] = v
    return result


# ------------------------------------------------------------
# Outcomes averaged per arm: name -> (engine key, scale)
# ------------------------------------------------------------
OUTCOME_KEYS = {
    "cost_base":    ("cost_base", 1.0),
    "cost_dsa1":    ("cost_dsa1", 1.0),
    "health_cost":  ("health_cost", 1.0),
    "housing_base": ("housing_cost_base", 1.0),
    "housing_dsa1": ("housing_cost_dsa1", 1.0),
    "qaly_base":    ("qaly_base", 1.0),
    "qaly_dsa2":    ("qaly_dsa2", 1.0),
    "years_H":      ("months_H", 1 / 12),
    "years_S":      ("months_S", 1 / 12),
    "years_U":      ("months_U", 1 / 12),
    "years_YT":     ("months_YT", 1 / 12),
    "years_AT":     ("months_AT", 1 / 12),
    "age_death":    ("age_at_death", 1.0),
    "deaths":       ("deaths", 1.0),
    "od_fatal":     ("od_fatal", 1.0),
    "od_nonfatal":  ("od_nonfatal", 1.0),
    "od_total":     ("od_total", 1.0),
}
for _hlab in v_housing:   # overdoses by housing state at the event
    for _kind in ("fatal", "nonfatal", "total"):
        OUTCOME_KEYS[f"od_{_kind}_{_hlab}"] = (f"od_{_kind}_{_hlab}", 1.0)
for _ak in ["acc_started_U", "acc_ever_S", "acc_ever_H", "acc_alive_12mo",
            "acc_ever_S_alive", "acc_ever_H_alive"]:   # housing validation
    OUTCOME_KEYS[_ak] = (_ak, 1.0)


def stack_cohort(n_reps):
    """Tile the baseline cohort n_reps times for one vectorised pass."""
    return (np.tile(init_age, n_reps), np.tile(init_sex_int, n_reps),
            np.tile(init_housing_int, n_reps), np.tile(init_OUD_int, n_reps))


def run_arm_chunked(arm, housing_trans, seed, n_reps, reps_per_chunk, **kw):
    """Simulate one arm in replication chunks and return its cohort means.

    Keys are `<arm>_<outcome>`. When replication tracking is on, the record also
    holds replication-level values (`rep__<arm>__<outcome>`) and their standard
    deviation (`<arm>_repsd_<outcome>`).
    """
    track = SAVE_PER_REPLICATION and (REP_TRACK_ARMS is None or arm in REP_TRACK_ARMS)
    totals, n_total = {}, 0
    rep_means = {o: [] for o in CHUNK_TRACK} if track else {}
    for ci, start in enumerate(range(0, n_reps, reps_per_chunk)):
        k = min(reps_per_chunk, n_reps - start)
        ages, sexes, houses, ouds = stack_cohort(k)
        res = run_microsim(ages, sexes, houses, ouds, housing_trans=housing_trans,
                           seed=seed * 1000 + ci, **kw)
        for name, (key, scale) in OUTCOME_KEYS.items():
            totals[name] = totals.get(name, 0.0) + \
                float(np.asarray(res[key], float).sum()) * scale
        for o in rep_means:
            key, scale = OUTCOME_KEYS[o]
            v = np.asarray(res[key], float).reshape(k, -1).mean(axis=1)
            rep_means[o].extend((v * scale).tolist())
        n_total += ages.size

    out = {f"{arm}_{n}": v / n_total for n, v in totals.items()}
    for o, vals in rep_means.items():
        out[f"rep__{arm}__{o}"] = vals
        out[f"{arm}_repsd_{o}"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan
    return out

## 12. Arms, scenarios and run fingerprint

Every scenario pairs a comparator arm with an intervention arm and selects the cost and utility variants to read; all scenarios are evaluated from the same simulated arms.

In [ ]:
def _thr_arm(M):
    """Arm name for threshold multiplier M, e.g. 1.0 -> 'thr1p0'."""
    return f"thr{str(M).replace('.', 'p')}"


ARMS = ["sq"] + [_thr_arm(M) for M in THRESHOLD_ARM_M]
ARM_EFFECT = {"sq": 0.0, **{_thr_arm(M): M for M in THRESHOLD_ARM_M}}
INT_ARM = _thr_arm(THRESHOLD_MEASURE_A)                     # Measure A
ENH_ARM = _thr_arm(DSA3_EFFECT_MULTIPLIER) if RUN_DSA3_PLACEMENT else None

ARM_LABELS = {"sq": "Status quo",
              **{_thr_arm(M): (f"Measure A (M = {M:.2f})" if M == THRESHOLD_MEASURE_A
                               else f"Housing effect M = {M:.2f}")
                 for M in THRESHOLD_ARM_M}}

# 1-year housing-access targets and calibrated deterministic rates per arm
ARM_TARGETS = {"sq": TARGET_SQ}
for _M in THRESHOLD_ARM_M:
    ARM_TARGETS[_thr_arm(_M)] = tuple(float(min(sq + _M * (it - sq), 0.999))
                                      for sq, it in zip(TARGET_SQ, TARGET_INT))

ARM_RATES, ARM_TRANS = {}, {}
for _arm, _tgt in ARM_TARGETS.items():
    if _arm == "sq":
        _rus, _ruh = R_U_TO_S_SQ, R_U_TO_H_SQ
    else:
        _rus, _ruh, _res = calibrate_unsheltered_rates(
            _tgt[0], _tgt[1], R_S_TO_U, R_S_TO_H, R_H_TO_U)
        assert _res < 1e-12, f"{_arm}: calibration did not converge"
    ARM_RATES[_arm] = (_rus, _ruh, R_S_TO_U, R_S_TO_H, R_H_TO_U)
    ARM_TRANS[_arm] = build_housing_trans(*ARM_RATES[_arm])
    assert np.allclose(ARM_TRANS[_arm].sum(axis=2), 1.0) and (ARM_TRANS[_arm] >= 0).all()

# ------------------------------------------------------------
# Scenarios
# ------------------------------------------------------------
SCENARIOS = [
    {"key": "base_case", "label": "Base case",
     "comparator": "sq", "intervention": INT_ARM, "cost": "base", "qaly": "base",
     "note": "Measure A housing intervention vs status quo"},
    {"key": "dsa_housing_cost", "label": "Doubled permanent housing cost",
     "comparator": "sq", "intervention": INT_ARM, "cost": "dsa1", "qaly": "base",
     "note": f"permanent housing at ${c_housing_H * DSA1_HOUSING_MULT:,.0f}/yr"},
    {"key": "dsa_qol", "label": "Higher unhoused quality of life",
     "comparator": "sq", "intervention": INT_ARM, "cost": "base", "qaly": "dsa2",
     "note": f"q_PEH = {Q_PEH_DSA2} instead of {Q_PEH_BASE}"},
]
if ENH_ARM is not None:
    SCENARIOS.append(
        {"key": "dsa_placement", "label": "Higher chance of housing placement",
         "comparator": "sq", "intervention": ENH_ARM, "cost": "base", "qaly": "base",
         "note": (f"housing-access gain scaled {DSA3_EFFECT_MULTIPLIER}x "
                  f"relative to Measure A")})
SCENARIO_BY_KEY = {s["key"]: s for s in SCENARIOS}


def scenario_columns(scn, arm_role):
    """Column names of a scenario's 'comparator' or 'intervention' arm."""
    arm = scn[arm_role]
    cols = {"cost": f"{arm}_cost_{scn['cost']}",
            "housing_cost": f"{arm}_housing_{scn['cost']}",
            "qaly": f"{arm}_qaly_{scn['qaly']}"}
    for stem in ["health_cost", "years_H", "years_S", "years_U", "years_YT",
                 "years_AT", "age_death", "od_fatal", "od_nonfatal", "od_total"]:
        cols[stem] = f"{arm}_{stem}"
    return cols


# Arms simulated by this job (a subset only when PSA_ARMS is set)
RUN_ARMS = ([a for a in ARMS if a in PSA_ARM_SUBSET.split(",")]
            if PSA_ARM_SUBSET else list(ARMS))
assert RUN_ARMS, f"PSA_ARMS={PSA_ARM_SUBSET!r} matched none of {ARMS}"

print(pd.DataFrame([
    {"Arm": a, "Label": ARM_LABELS[a], "M": ARM_EFFECT[a],
     "1-yr interim": ARM_TARGETS[a][0], "1-yr permanent": ARM_TARGETS[a][1],
     "r_U->S": ARM_RATES[a][0], "r_U->H": ARM_RATES[a][1]}
    for a in ARMS]).to_string(index=False, float_format=lambda x: f"{x:.5f}"))
print(f"\n{len(SCENARIOS)} scenarios from {len(ARMS)} simulated arms:")
for _s in SCENARIOS:
    print(f"  {_s['label']:34s} {_s['intervention']} vs {_s['comparator']} "
          f"| cost={_s['cost']}, qaly={_s['qaly']} | {_s['note']}")
if RUN_ARMS != ARMS:
    print(f"This job simulates {len(RUN_ARMS)} of {len(ARMS)} arms: {RUN_ARMS}")

In [ ]:
# ------------------------------------------------------------
# Run fingerprint: a hash of every headline input. Checkpoints store it, so a
# resumed or merged run with a different parameterisation fails loudly.
# ------------------------------------------------------------
RUN_CONFIG = {
    "model_version": MODEL_VERSION,
    "n_i": n_i, "n_psa": n_psa,
    "n_reps_psa": n_reps_psa, "max_cycles": max_cycles,
    "d_c": round(d_c, 10), "d_e": round(d_e, 10),
    "seeds": [COHORT_SEED, PSA_PARAM_SEED, PSA_SIM_SEED],
    "housing_raw": [RAW_INTERIM_N, RAW_INTERIM_DENOM, RAW_PERM_N,
                    RAW_S_TO_U_N, RAW_S_TO_U_DENOM, RAW_S_TO_H_N,
                    RAW_S_TO_H_DENOM, RAW_H_TO_U_P],
    "housing_beta": [BETA_INTERIM_1YR, BETA_PERM_1YR, BETA_S_TO_U_PROP,
                     BETA_S_TO_H_PROP, BETA_H_TO_U_1YR],
    "housing_targets": [round(x, 8) for t in ARM_TARGETS.values() for x in t],
    "threshold_grid": THRESHOLD_GRID,
    "housing_rates": [round(x, 8) for a in ARMS for x in ARM_RATES[a][:2]]
                     + [round(R_S_TO_U, 8), round(R_S_TO_H, 8), round(R_H_TO_U, 8)],
    "ma_ac": MA_AC_BASE, "ma_od": MA_OD_BASE,
    "background_anchor": BACKGROUND_ANCHOR,
    "home_mult": [b[2] for b in HOME_MULT_BANDS],
    "sample_home_mult": SAMPLE_HOME_MULT,
    "total_od": TOTAL_OD_BASE.round(8).tolist(),
    "p_fatal": [P_FATAL_HOUSED_BASE, P_FATAL_SHELTERED_BASE,
                P_FATAL_UNSHELTERED_BASE, PF_EFFECTIVE_N],
    "od_rates": [OD_RATE_YT, OD_RATE_NT],
    "housing_cost_cv": HOUSING_COST_CV,
    "pf_uniform": PF_UNHOUSED_UNIFORM,
    "c_health_AT": c_health_AT,
    "costs": [c_health_YT, c_health_NT, c_health_S, c_health_U, c_treatment_YT,
              c_per_OD_H, c_per_OD_S, c_per_OD_U, c_housing_H,
              c_emergency_shelter, c_transitional_shelter],
    "cost_year": COST_YEAR,
    "deflators": [round(DEFL_BG_HEALTH_2019, 8), round(DEFL_HEALTH_2024, 8),
                  round(DEFL_HOUSING_2024, 8)],
    "raw_costs": {k: v[0] for k, v in RAW_COSTS.items()},
    "q_peh": [Q_PEH_BASE, Q_PEH_DSA2],
    "dsa": [DSA1_HOUSING_MULT, DSA3_EFFECT_MULTIPLIER],
    "life_table_is_prob": LIFE_TABLE_IS_PROBABILITY,
    "life_table_col": LIFE_TABLE_VALUE_COL,
    "scenarios": [(s["key"], s["comparator"], s["intervention"],
                   s["cost"], s["qaly"]) for s in SCENARIOS],
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, default=_json_default).encode()
).hexdigest()[:16]

print(f"Run fingerprint: {RUN_FINGERPRINT} (model {MODEL_VERSION})")

## 13. Probabilistic sensitivity analysis

Each draw samples a full parameter set, rebuilds every model input, recalibrates the housing rates and simulates all arms with `n_reps_psa` replications. Arms within a draw share the parameter values and random-number seeds, so incremental results are paired.

| Parameter group | PSA distribution |
|---|---|
| Background mortality; age/sex utility multipliers and background costs | Fixed |
| Ma et al. all-cause and overdose mortality (6 rates) | Gamma; non-overdose excess derived per draw |
| Homelessness mortality multipliers | Uniform, ±20% (modeller-assigned) |
| Overdose incidence (YT, NT) | Lognormal fitted to the reported 95% CI; shared across housing states |
| Overdose case fatality (H, S, U) | Beta fitted to each reported survival CI |
| Housing transitions | Beta on LA County data, then per-draw recalibration |
| OUD transition rates | Lognormal fitted to the reported 95% CI |
| Costs | Gamma fitted to the published 95% CI; CV = 20% for housing |
| State utilities | Beta |

Lognormal and Gamma fits hold the mean at the reported point estimate. Draw-level ICERs are not summarised because the ratio is unstable as ΔQALY approaches zero; decision uncertainty is summarised with INMB(λ) = λ·ΔQALY − ΔCost, the CEAC and the EVPI.

### 13.1 Parameter sampling

In [ ]:
def _lognormal_mean_params(mean, lo, hi):
    """(mu, sigma) of a lognormal with E[X] = mean fitted to the 95% CI."""
    s = fit_lognormal_sigma(mean, lo, hi)
    return float(np.log(mean) - s * s / 2), s


# Overdose incidence by treatment status
OD_PSA = {k: _lognormal_mean_params(*v) for k, v in OD_RATE_REPORTED.items()}

# OUD transition rates, (mean, 95% CI low, 95% CI high). Several intervals are
# symmetric on the arithmetic rather than the log scale, so the lognormal fit
# preserves the mean exactly and approximates the interval.
OUD_TRANS_REPORTED = {
    "r_NT_to_YT_H":       (0.426,  0.367,  0.489),
    "r_NT_to_YT_S":       (0.221,  0.191,  0.254),
    "r_NT_to_YT_U":       (0.211,  0.182,  0.242),
    "r_NT_to_AT_H":       (0.0791, 0.0040, 0.1549),
    "r_NT_to_AT_S":       (0.0074, 0.0040, 0.0144),
    "r_NT_to_AT_U":       (0.0074, 0.0040, 0.0144),
    "r_YT_to_NT_H":       (1.125,  0.634,  1.830),
    "r_YT_to_NT_S":       (1.531,  0.860,  2.500),
    "r_YT_to_NT_U":       (1.531,  0.860,  2.500),
    "r_YT_to_AT_H":       (0.316,  0.296,  0.337),
    "r_YT_to_AT_S":       (0.029,  0.028,  0.031),
    "r_YT_to_AT_U":       (0.029,  0.028,  0.031),
    "r_AT_to_NT_first_H": (0.344,  0.300,  0.390),
    "r_AT_to_NT_first_S": (0.462,  0.404,  0.524),
    "r_AT_to_NT_first_U": (0.462,  0.404,  0.524),
    "r_AT_to_NT_y10_H":   (0.023,  0.0047, 0.0419),
    "r_AT_to_NT_y10_S":   (0.031,  0.0063, 0.0564),
    "r_AT_to_NT_y10_U":   (0.031,  0.0063, 0.0564),
}
OUD_TRANS_PSA = {k: _lognormal_mean_params(*v) for k, v in OUD_TRANS_REPORTED.items()}

# Check: each fitted distribution is centred on the reported mean, which is also
# the deterministic rate used by the model
_DET_RATES = {}
for _i, _h in enumerate(v_housing):
    for _name, _arr in [("NT_to_YT", R_NT_TO_YT), ("NT_to_AT", R_NT_TO_AT),
                        ("YT_to_NT", R_YT_TO_NT), ("YT_to_AT", R_YT_TO_AT),
                        ("AT_to_NT_first", R_AT_TO_NT_FIRST),
                        ("AT_to_NT_y10", R_AT_TO_NT_Y10)]:
        _DET_RATES[f"r_{_name}_{_h}"] = _arr[_i]
_n_approx = 0
for _k, (_m, _lo, _hi) in OUD_TRANS_REPORTED.items():
    _mu, _s = OUD_TRANS_PSA[_k]
    assert abs(np.exp(_mu + _s * _s / 2) / _m - 1) < 1e-6, f"{_k}: PSA mean off"
    assert abs(_DET_RATES[_k] / _m - 1) < 1e-6, f"{_k}: deterministic rate differs"
    _fl, _fh = lognorm.ppf([0.025, 0.975], _s, scale=np.exp(_mu))
    _n_approx += not (abs(_fl / _lo - 1) < 0.15 and abs(_fh / _hi - 1) < 0.15)


def _ln(rng, mu, sigma):
    return float(rng.lognormal(mean=mu, sigma=sigma))


def _gam(rng, shape, scale):
    return float(rng.gamma(shape=shape, scale=scale))


def _bet(rng, a, b):
    return float(rng.beta(a, b))


def draw_psa_params(draw_id):
    """Sample one complete parameter set. The sampling order must not change."""
    rng = np.random.default_rng(PSA_PARAM_SEED + draw_id)
    p = {"draw_id": draw_id}

    # OUD mortality (Ma et al.): all-cause and overdose CMR; the overdose rates
    # are used only to derive the non-overdose excess
    p["ma_ac_YT"] = _gam(rng, 208.10, 0.000044210)
    p["ma_od_YT"] = _gam(rng, 138.30, 0.000017354)
    p["ma_ac_NT"] = _gam(rng,  50.78, 0.000962994)
    p["ma_od_NT"] = _gam(rng,  44.37, 0.000547638)
    p["ma_ac_PC"] = _gam(rng, 226.69, 0.000074550)
    p["ma_od_PC"] = _gam(rng, 113.69, 0.000059814)
    p["nonod_excess_YT"] = max(p["ma_ac_YT"] - p["ma_od_YT"] - BACKGROUND_ANCHOR, 0.0)
    p["nonod_excess_NT"] = max(p["ma_ac_NT"] - p["ma_od_NT"] - BACKGROUND_ANCHOR, 0.0)
    p["nonod_excess_AT"] = max(p["ma_ac_PC"] - p["ma_od_PC"] - BACKGROUND_ANCHOR, 0.0)

    # Homelessness mortality multipliers
    for i, (_, _, base, low, high) in enumerate(HOME_MULT_BANDS):
        p[f"home_mult_{i}"] = (float(rng.uniform(low, high)) if SAMPLE_HOME_MULT
                               else float(base))

    # Overdose incidence and case fatality
    p["od_YT"] = _ln(rng, *OD_PSA["YT"])
    p["od_NT"] = _ln(rng, *OD_PSA["NT"])
    for h in v_housing:
        p[f"p_fatal_{h}"] = _bet(rng, *PF_BETA[h])

    # Housing: LA County proportions
    p["p_interim_1yr"] = _bet(rng, *BETA_INTERIM_1YR)
    p["p_perm_1yr"]    = _bet(rng, *BETA_PERM_1YR)
    p["p_S_to_U_prop"] = _bet(rng, *BETA_S_TO_U_PROP)
    p["p_S_to_H_prop"] = _bet(rng, *BETA_S_TO_H_PROP)
    p["p_H_to_U_1yr"]  = _bet(rng, *BETA_H_TO_U_1YR)

    # OUD transition rates
    for k, (mu, s) in OUD_TRANS_PSA.items():
        p[k] = _ln(rng, mu, s)

    # Costs, COST_YEAR USD
    for k, (shape, scale) in COST_PSA.items():
        p[k] = _gam(rng, shape, scale)

    # State utilities
    p["q_H_YT"]       = _bet(rng,  1262.7,   488.6)
    p["q_S_YT"]       = _bet(rng,  2144.3,  1826.6)
    p["q_U_YT"]       = _bet(rng,  3528.5, 10932.5)
    p["q_H_NT"]       = _bet(rng,  3641.7,  1809.9)
    p["q_S_NT"]       = _bet(rng,  5320.3,  5320.3)
    p["q_U_NT"]       = _bet(rng,  7499.5, 25684.1)
    p["q_H_AT_first"] = _bet(rng,  1501.6,   508.6)
    p["q_S_AT_first"] = _bet(rng,  2700.4,  2130.4)
    p["q_U_AT_first"] = _bet(rng,  4347.3, 12835.8)
    p["q_H_AT_y10"]   = _bet(rng,   345.3,     6.0)
    p["q_S_AT_y10"]   = _bet(rng,  5493.0,  1970.3)
    p["q_U_AT_y10"]   = _bet(rng, 13967.8, 28104.0)
    return p


# ------------------------------------------------------------
# Model inputs from a parameter draw
# ------------------------------------------------------------
def psa_mortality_inputs(p):
    """(OUD non-overdose excess [YT, NT, AT], homelessness multiplier lookup)."""
    oud_nonod = np.array([p["nonod_excess_YT"], p["nonod_excess_NT"],
                          p["nonod_excess_AT"]])
    home = build_home_mult_lut([p[f"home_mult_{i}"]
                                for i in range(len(HOME_MULT_BANDS))])
    return oud_nonod, home


def psa_od_matrices(p):
    """Total, fatal and nonfatal overdose rate matrices and p_fatal."""
    total = np.tile([p["od_YT"], p["od_NT"], 0.0], (3, 1))
    return build_od_matrices(total, [p[f"p_fatal_{h}"] for h in v_housing])


def psa_housing_trans(p):
    """Recalibrated housing transition matrices for every arm, plus diagnostics.

    The status quo is calibrated to this draw's sampled 1-year probabilities;
    each intervention arm to its fixed target.
    """
    r_SU, r_SH = sheltered_rates(p["p_S_to_U_prop"], p["p_S_to_H_prop"])
    r_HU = housed_exit_rate(p["p_H_to_U_1yr"])

    out, diag = {}, {}
    for arm in ARMS:
        targets = ((p["p_interim_1yr"], p["p_perm_1yr"]) if arm == "sq"
                   else ARM_TARGETS[arm])
        r_US, r_UH, cost = calibrate_unsheltered_rates(
            targets[0], targets[1], r_SU, r_SH, r_HU)
        out[arm] = build_housing_trans(r_US, r_UH, r_SU, r_SH, r_HU)
        diag[f"cal_{arm}_r_U_to_S"] = r_US
        diag[f"cal_{arm}_r_U_to_H"] = r_UH
        diag[f"cal_{arm}_residual"] = cost
    diag["cal_r_S_to_U"] = float(r_SU)
    diag["cal_r_S_to_H"] = float(r_SH)
    diag["cal_r_H_to_U"] = float(r_HU)
    return out, diag


def psa_cost_matrices(p, total_od_mat):
    """State health care cost matrix and housing cost variants for one draw."""
    health = build_health_cost_matrix(
        total_od_mat,
        c_od=(p["c_per_OD_H"], p["c_per_OD_S"], p["c_per_OD_U"]),
        c_YT=p["c_health_YT"], c_NT=p["c_health_NT"], c_AT=0.0,
        c_S=p["c_health_S"], c_U=p["c_health_U"], c_tx=p["c_treatment_YT"])
    c_shelt = (p_emergency_shelter * p["c_emergency_shelter"]
               + p_transitional_shelter * p["c_transitional_shelter"])
    housing = {
        "base": build_housing_cost_matrix(p["c_housing_H"], c_shelt),
        "dsa1": build_housing_cost_matrix(p["c_housing_H"] * DSA1_HOUSING_MULT, c_shelt),
    }
    return health, housing


def psa_qaly_matrices(p):
    """Base and higher-QoL (dsa2) utility matrices for one draw.

    `base` uses the sampled state utilities. `dsa2` rescales the sheltered and
    unsheltered rows by Q_PEH_DSA2 / Q_PEH_BASE, keeping the draw's housed
    utilities and its sheltered-to-unsheltered ratio.
    """
    qm_b, atu_b = build_qaly_params(utilities=p)
    scale = Q_PEH_DSA2 / Q_PEH_BASE
    qm_d, atu_d = qm_b.copy(), atu_b.copy()
    for row in (S_IDX, U_IDX):
        qm_d[row] = np.clip(qm_b[row] * scale, 0.0, 1.0)
        atu_d[row] = np.clip(atu_b[row] * scale, 0.0, 1.0)
    return {"base": qm_b, "dsa2": qm_d}, {"base": atu_b, "dsa2": atu_d}


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------
# The Ma decomposition should reproduce the central excess rates on average
_chk = pd.DataFrame([draw_psa_params(i) for i in range(400)])
for _col, _want in [("nonod_excess_YT", 0.0040), ("nonod_excess_NT", 0.0218),
                    ("nonod_excess_AT", 0.0073)]:
    assert abs(_chk[_col].mean() / _want - 1) < 0.10, f"{_col} drifted"

_p0 = draw_psa_params(0)
_, _diag0 = psa_housing_trans(_p0)
print(f"{len(_p0) - 1} parameters per draw "
      f"(homelessness multipliers sampled: {SAMPLE_HOME_MULT})")
print(f"OUD transition lognormals: all centred on the reported means; "
      f"{_n_approx} of {len(OUD_TRANS_REPORTED)} fitted CIs differ from the "
      f"reported CI by > 15% at a bound")
print("Mean non-overdose excess over 400 draws (YT, NT, AT): "
      + ", ".join(f"{_chk[c].mean():.5f}" for c in
                  ["nonod_excess_YT", "nonod_excess_NT", "nonod_excess_AT"]))
print("Draw 0 calibrated r_U->S by arm: "
      + ", ".join(f"{a} {_diag0[f'cal_{a}_r_U_to_S']:.4f}" for a in ARMS))
print(f"Draw 0 max calibration residual: "
      f"{max(v for k, v in _diag0.items() if k.endswith('residual')):.2e}")

### 13.2 Run the PSA

The long stage. Results are checkpointed per batch of draws; re-running the cell resumes from the last completed batch. `Checkpoint(f"psa_{SHARD_TAG}").reset()` discards this job's results.

In [ ]:
def psa_draw_worker(draw_id):
    """Sample one draw, rebuild all inputs, simulate every arm, return one record."""
    p = draw_psa_params(draw_id)

    oud_nonod, home_lut = psa_mortality_inputs(p)
    total_od, fatal_od, nonfatal_od, p_fatal = psa_od_matrices(p)
    health_mat, housing_mats = psa_cost_matrices(p, total_od)
    qaly_mats, at_util_mats = psa_qaly_matrices(p)
    trans, cal_diag = psa_housing_trans(p)
    oud_trans = build_OUD_trans(psa_rates=p)
    at_relapse = at_relapse_from_psa(p)

    sim_seed = PSA_SIM_SEED + draw_id
    rec = {"unit_id": draw_id, "draw_id": draw_id,
           "seed": sim_seed, "param_seed": PSA_PARAM_SEED + draw_id,
           "model_version": MODEL_VERSION}

    kw = dict(
        d_c=d_c, d_e=d_e,
        health_cost_mat=health_mat, housing_cost_mats=housing_mats,
        qaly_mats=qaly_mats, at_util_mats=at_util_mats,
        baseline_cost_lut=baseline_cost_lut, qaly_mult_lut=qaly_age_mult,
        bg_mort_lut=bg_mort_lut, home_mult_lut=home_lut,
        oud_nonod=oud_nonod, fatal_od_mat=fatal_od, nonfatal_od_mat=nonfatal_od,
        OUD_trans=oud_trans, at_relapse=at_relapse, max_cycles=max_cycles,
    )
    for arm in RUN_ARMS:
        rec.update(run_arm_chunked(arm, trans[arm], sim_seed,
                                   n_reps_psa, PSA_REPS_PER_CHUNK, **kw))
        tot = rec[f"{arm}_od_total"]
        rec[f"{arm}_fatal_fraction"] = rec[f"{arm}_od_fatal"] / tot if tot > 0 else np.nan

    # Paired incremental outcomes (skipped when an arm-sharded job lacks one arm;
    # the analysis recomputes them from the arm columns)
    for scn in SCENARIOS:
        c = scenario_columns(scn, "comparator")
        i = scenario_columns(scn, "intervention")
        if c["cost"] in rec and i["cost"] in rec:
            rec[f"inc_{scn['key']}_cost"] = rec[i["cost"]] - rec[c["cost"]]
            rec[f"inc_{scn['key']}_qaly"] = rec[i["qaly"]] - rec[c["qaly"]]

    # Analytic 1-year housing-access references for the internal validation
    r_SU, r_SH, r_HU = (cal_diag["cal_r_S_to_U"], cal_diag["cal_r_S_to_H"],
                        cal_diag["cal_r_H_to_U"])
    for arm in RUN_ARMS:
        tS, tH = ((p["p_interim_1yr"], p["p_perm_1yr"]) if arm == "sq"
                  else ARM_TARGETS[arm])
        rus = cal_diag[f"cal_{arm}_r_U_to_S"]
        ruh = cal_diag[f"cal_{arm}_r_U_to_H"]
        rec[f"val_{arm}_target_S"], rec[f"val_{arm}_target_H"] = tS, tH
        rec[f"val_{arm}_ct_S"], rec[f"val_{arm}_ct_H"] = one_year_access_probs(
            rus, ruh, r_SU, r_SH, r_HU, t=1.0)
        rec[f"val_{arm}_monthly_S"], rec[f"val_{arm}_monthly_H"] = monthly_access_probs(
            rus, ruh, r_SU, r_SH, r_HU, months=HOUSING_VALID_MONTHS)

    rec.update(cal_diag)
    for _i, _h in enumerate(v_housing):
        rec[f"p_fatal_{_h}"] = float(p_fatal[_i])
    if SAVE_PSA_PARAMS:
        rec.update({f"par_{k}": v for k, v in p.items() if k != "draw_id"})
    return rec


print(f"PSA: {PSA_DRAW_END - PSA_DRAW_START:,} draws x {n_reps_psa} replications x "
      f"{len(RUN_ARMS)} arms, {n_workers} workers, checkpoint every "
      f"{PSA_BATCH_SIZE} draws\n")

# Skip draws already completed by any shard in this results directory, so a
# final full-range run only merges and analyses
_done_elsewhere = set()
for _p in sorted(glob.glob(rpath("checkpoints", "psa_*.jsonl"))):
    _done_elsewhere |= Checkpoint(os.path.basename(_p)[:-6]).completed_ids()
_todo = [d for d in range(PSA_DRAW_START, PSA_DRAW_END) if d not in _done_elsewhere]
if len(_todo) < PSA_DRAW_END - PSA_DRAW_START:
    print(f"{PSA_DRAW_END - PSA_DRAW_START - len(_todo):,} draws already done in "
          f"other checkpoints; {len(_todo):,} to simulate.")

df_psa = run_checkpointed(
    name=f"psa_{SHARD_TAG}", unit_ids=_todo, worker=psa_draw_worker,
    batch_size=PSA_BATCH_SIZE, n_workers=n_workers, desc="draws",
    fingerprint=RUN_FINGERPRINT)

_resid_cols = [c for c in df_psa.columns if c.endswith("_residual")]
if _resid_cols:
    _worst = df_psa[_resid_cols].to_numpy().max()
    print(f"Worst housing calibration residual in this job: {_worst:.2e}"
          + ("  WARNING: a calibration did not converge cleanly" if _worst > 1e-10 else ""))

## 14. Merge shards

Combines every `psa_*.jsonl` checkpoint in the results directory and checks that all shards share the current run fingerprint. The analysis below runs only when every draw and arm is present and `PEH_OUD_SKIP_ANALYSIS` is off; otherwise each analysis cell skips itself.

In [ ]:
def load_all_shards():
    """Combine every PSA shard checkpoint. Returns (df, shard table, merge mode)."""
    paths = sorted(glob.glob(rpath("checkpoints", "psa_*.jsonl")))
    if not paths:
        raise FileNotFoundError(f"no psa_*.jsonl in {rpath('checkpoints', '')}")

    frames, rows = [], []
    for p in paths:
        ck = Checkpoint(os.path.basename(p)[:-6])
        recs = ck.records()
        if not recs:
            continue
        df = pd.DataFrame(recs)
        frames.append(df)
        rows.append({"Shard": os.path.basename(p), "Records": len(df),
                     "Draws": f"{df.draw_id.min()}-{df.draw_id.max()}",
                     "Arms": sum(f"{a}_cost_base" in df.columns for a in ARMS),
                     "Fingerprint": (ck.read_meta() or {}).get("fingerprint")})
    raw = pd.concat(frames, ignore_index=True, sort=False)

    if len(raw) == raw["draw_id"].nunique():
        out = raw.sort_values("draw_id").reset_index(drop=True)
        mode = "draw shards, disjoint rows"
    else:
        # Arm shards: the same draw_ids with complementary columns
        out = (raw.groupby("draw_id", as_index=False)
                  .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
                  .sort_values("draw_id").reset_index(drop=True))
        mode = "arm shards, merged by draw_id"
    return out, pd.DataFrame(rows), mode


def extract_replications(df):
    """Split replication-level list columns out of the draw-level frame.

    Returns (df without list columns, long frame of draw_id, arm, rep_id, outcomes).
    Keeping lists out of df_psa lets it be written as parquet.
    """
    rep_cols = [c for c in df.columns if c.startswith("rep__")]
    if not rep_cols:
        return df, None

    by_arm = {}
    for c in rep_cols:
        _, arm, outcome = c.split("__", 2)
        by_arm.setdefault(arm, {})[outcome] = c

    frames = []
    for arm, outcomes in by_arm.items():
        sub = df[["draw_id"] + list(outcomes.values())].dropna(
            subset=list(outcomes.values()))
        if not len(sub):
            continue
        recs = {"draw_id": [], "arm": [], "rep_id": [], **{o: [] for o in outcomes}}
        for _, row in sub.iterrows():
            vals = {o: list(row[c]) for o, c in outcomes.items()}
            n = len(next(iter(vals.values())))
            recs["draw_id"].extend([int(row["draw_id"])] * n)
            recs["arm"].extend([arm] * n)
            recs["rep_id"].extend(range(n))
            for o in outcomes:
                recs[o].extend(vals[o])
        frames.append(pd.DataFrame(recs))

    long = (pd.concat(frames, ignore_index=True)
              .sort_values(["draw_id", "arm", "rep_id"]).reset_index(drop=True)
            if frames else None)
    return df.drop(columns=rep_cols), long


df_psa, table_shards, _mode = load_all_shards()
df_psa, df_reps = extract_replications(df_psa)

print(table_shards.to_string(index=False))
print(f"Merge mode: {_mode}; {len(df_psa):,} unique draws, {df_psa.shape[1]} columns")
if df_reps is not None:
    print(f"Replication-level rows: {len(df_reps):,}")

# ------------------------------------------------------------
# Consistency and completeness
# ------------------------------------------------------------
_fps = sorted(set(table_shards["Fingerprint"].dropna()))
if _fps and _fps != [RUN_FINGERPRINT]:
    raise StaleCheckpointError(
        f"Shard fingerprints {_fps} do not match the current run "
        f"({RUN_FINGERPRINT}); results from different parameterisations must "
        f"not be pooled.")

_ids = df_psa["draw_id"].to_numpy()
_gaps = np.setdiff1d(np.arange(n_psa), _ids)
_arms_done = [a for a in ARMS if f"{a}_cost_base" in df_psa.columns]
_incomplete = [a for a in _arms_done if df_psa[f"{a}_cost_base"].isna().any()]

_RESULTS_COMPLETE = len(_gaps) == 0 and len(_arms_done) == len(ARMS) and not _incomplete
RUN_ANALYSIS = _RESULTS_COMPLETE and not SKIP_ANALYSIS

print(f"\nDraws: {len(_ids):,} of {n_psa:,} | arms: {len(_arms_done)} of {len(ARMS)} "
      f"| arms with gaps: {len(_incomplete)}")
if RUN_ANALYSIS:
    print("Complete: running the analysis.")
elif _RESULTS_COMPLETE:
    print("Complete, but PEH_OUD_SKIP_ANALYSIS is set: analysis skipped in this job.")
else:
    print("Incomplete (a shard of a parallel run): analysis skipped. Run the final "
          "merge job once every shard has finished.")
    if len(_gaps):
        print(f"  missing draws: {_gaps[:12]}{' ...' if len(_gaps) > 12 else ''}")
    if len(_arms_done) < len(ARMS):
        print(f"  missing arms : {[a for a in ARMS if a not in _arms_done]}")


def money(x):
    """Format a dollar amount; '--' for non-finite values."""
    if not np.isfinite(x):
        return "--"
    return f"-${abs(x):,.0f}" if x < 0 else f"${x:,.0f}"


def analysis_gate(section):
    """True if the analysis should run; otherwise print why `section` is skipped."""
    if not RUN_ANALYSIS:
        print(f"[{section}] skipped: PSA results incomplete or analysis disabled "
              f"({len(_ids):,}/{n_psa:,} draws, {len(_arms_done)}/{len(ARMS)} arms).")
    return RUN_ANALYSIS

# Analysis

## 15. PSA results

Each scenario is summarised by PSA means with 95% uncertainty intervals, followed by incremental outcomes, the ICER, INMB and the probability of cost-effectiveness.

In [ ]:
if analysis_gate("PSA results"):
    UI_LO, UI_HI = 2.5, 97.5

    def ui(values, scale=1.0):
        """PSA mean and 95% uncertainty interval (percentiles across draws)."""
        v = np.asarray(values, dtype=float) * scale
        return v.mean(), np.percentile(v, UI_LO), np.percentile(v, UI_HI)

    def fatal_fraction(df, arm, state=None):
        """Simulated case fatality (fatal / total overdoses), overall or for one
        housing state; a ratio of means pooled over person-time."""
        suf = f"_{state}" if state else ""
        tot = df[f"{arm}_od_total{suf}"].mean()
        return df[f"{arm}_od_fatal{suf}"].mean() / tot if tot > 0 else np.nan

    # (label, column stem, scale, format)
    PSA_OUTCOME_SPECS = [
        ("Years in permanent housing",        "years_H",      1.0,    "{:.1f}"),
        ("Years sheltered",                   "years_S",      1.0,    "{:.1f}"),
        ("Years unsheltered",                 "years_U",      1.0,    "{:.1f}"),
        ("Years in OUD treatment",            "years_YT",     1.0,    "{:.1f}"),
        ("Years abstinent",                   "years_AT",     1.0,    "{:.1f}"),
        ("Total overdoses per person",        "od_total",     1.0,    "{:.1f}"),
        ("Nonfatal overdoses per person",     "od_nonfatal",  1.0,    "{:.1f}"),
        ("Fatal overdoses per 1,000 persons", "od_fatal",     1000.0, "{:.0f}"),
        ("Mean age at death (yrs)",           "age_death",    1.0,    "{:.1f}"),
        ("QALYs",                             "qaly",         1.0,    "{:.2f}"),
        ("Health care cost ($)",              "health_cost",  1.0,    "{:,.0f}"),
        ("Housing cost ($)",                  "housing_cost", 1.0,    "{:,.0f}"),
        ("Total cost ($)",                    "cost",         1.0,    "{:,.0f}"),
    ]

    # ------------------------------------------------------------
    # Incremental quantities per scenario
    # ------------------------------------------------------------
    psa_scenario = {}
    for scn in SCENARIOS:
        c = scenario_columns(scn, "comparator")
        i = scenario_columns(scn, "intervention")
        d_cost = (df_psa[i["cost"]] - df_psa[c["cost"]]).to_numpy()
        d_qaly = (df_psa[i["qaly"]] - df_psa[c["qaly"]]).to_numpy()
        psa_scenario[scn["key"]] = {
            "scn": scn, "cmp_cols": c, "int_cols": i,
            "delta_cost": d_cost, "delta_qaly": d_qaly,
            "icer": d_cost.mean() / d_qaly.mean() if d_qaly.mean() != 0 else np.nan,
            "inmb": {lam: lam * d_qaly - d_cost for lam in WTP_GRID},
            "ceac": np.array([((lam * d_qaly - d_cost) >= 0).mean() for lam in WTP_GRID]),
        }

    # ------------------------------------------------------------
    # Absolute outcomes by scenario and arm
    # ------------------------------------------------------------
    _disp, _num = [], []
    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        for role, cols in [("Comparator", S["cmp_cols"]), ("Intervention", S["int_cols"])]:
            d = {"Scenario": scn["label"], "Arm": role}
            nrow = dict(d)
            for lbl, stem, scale, fmt in PSA_OUTCOME_SPECS:
                m, lo, hi = ui(df_psa[cols[stem]], scale)
                d[lbl] = f"{fmt.format(m)} ({fmt.format(lo)}\u2013{fmt.format(hi)})"
                nrow.update({f"{lbl} | mean": m, f"{lbl} | lo": lo, f"{lbl} | hi": hi})
            _disp.append(d)
            _num.append(nrow)
    table_psa_absolute_display = pd.DataFrame(_disp)
    table_psa_absolute_numeric = pd.DataFrame(_num)

    # ------------------------------------------------------------
    # Incremental outcomes, ICER, INMB and P(cost-effective)
    # ------------------------------------------------------------
    _rows_d, _rows_n = [], []
    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        dc_m, dc_lo, dc_hi = ui(S["delta_cost"])
        dq_m, dq_lo, dq_hi = ui(S["delta_qaly"])
        d = {"Scenario": scn["label"],
             "Incremental cost ($)": f"{money(dc_m)} ({money(dc_lo)}\u2013{money(dc_hi)})",
             "Incremental QALYs": f"{dq_m:.3f} ({dq_lo:.3f}\u2013{dq_hi:.3f})",
             "ICER ($/QALY)": money(S["icer"])}
        n = {"Scenario": scn["label"],
             "delta_cost_mean": dc_m, "delta_cost_lo": dc_lo, "delta_cost_hi": dc_hi,
             "delta_qaly_mean": dq_m, "delta_qaly_lo": dq_lo, "delta_qaly_hi": dq_hi,
             "ICER": S["icer"]}
        for lam in WTP_REPORT:
            v = S["inmb"][lam]
            m, lo, hi = ui(v)
            k = f"${lam // 1000}k"
            d[f"INMB at {k} ($)"] = f"{money(m)} ({money(lo)}\u2013{money(hi)})"
            d[f"P(CE) at {k}"] = f"{(v >= 0).mean() * 100:.1f}%"
            n.update({f"inmb_{lam}_mean": m, f"inmb_{lam}_lo": lo,
                      f"inmb_{lam}_hi": hi, f"p_ce_{lam}": (v >= 0).mean()})
        _rows_d.append(d)
        _rows_n.append(n)
    table_psa_incremental_display = pd.DataFrame(_rows_d)
    table_psa_incremental_numeric = pd.DataFrame(_rows_n)

    df_ceac = pd.DataFrame({"WTP": WTP_GRID})
    for scn in SCENARIOS:
        df_ceac[scn["key"]] = psa_scenario[scn["key"]]["ceac"]

    print(f"PSA results: {len(df_psa):,} draws x {n_reps_psa} replications, "
          f"{COST_YEAR} USD, 95% uncertainty intervals\n")
    _show = ["Arm", "Years in permanent housing", "Fatal overdoses per 1,000 persons",
             "Mean age at death (yrs)", "QALYs", "Health care cost ($)",
             "Housing cost ($)", "Total cost ($)"]
    for scn in SCENARIOS:
        print("=" * 118)
        print(f"{scn['label']}  [{scn['note']}]")
        print(table_psa_absolute_display.loc[
            table_psa_absolute_display["Scenario"] == scn["label"], _show].to_string(index=False))
    print("=" * 118)
    print("\nIncremental results (ICER = mean incremental cost / mean incremental QALYs)")
    print("=" * 118)
    print(table_psa_incremental_display.to_string(index=False))
    print("=" * 118)

    # ------------------------------------------------------------
    # Overdose QA: simulated events vs intended case fatality
    # ------------------------------------------------------------
    # Simulated case fatality sits slightly below the intended value by design: a
    # fatal overdose is realised through the monthly death probability
    # 1 - exp(-lambda/12), marginally below the rate lambda/12 used for nonfatal
    # draws. The gap is second order and largest at the oldest ages; a gap above
    # about 1 percentage point would indicate a real inconsistency.
    _intended = dict(zip(v_housing, p_fatal_base))
    _qa_rows = []
    for arm in ARMS:
        row = {"Arm": arm,
               "Total OD/person": df_psa[f"{arm}_od_total"].mean(),
               "Nonfatal OD/person": df_psa[f"{arm}_od_nonfatal"].mean(),
               "Fatal OD/person": df_psa[f"{arm}_od_fatal"].mean(),
               "Deaths/person": df_psa[f"{arm}_deaths"].mean(),
               "Overall fatal frac": fatal_fraction(df_psa, arm)}
        for hlab in v_housing:
            row[f"Fatal frac {hlab}"] = fatal_fraction(df_psa, arm, hlab)
        row["Identity gap"] = abs(df_psa[f"{arm}_od_total"].mean()
                                  - df_psa[f"{arm}_od_fatal"].mean()
                                  - df_psa[f"{arm}_od_nonfatal"].mean())
        _qa_rows.append(row)
    table_od_qa = pd.DataFrame(_qa_rows)

    print("\nOverdose QA: simulated events and case fatality by housing state")
    print(table_od_qa.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("Intended case fatality: "
          + ", ".join(f"{k} {v:.4f}" for k, v in _intended.items()))

    # Warnings rather than assertions: small test runs breach the case-fatality
    # tolerance through Monte Carlo noise alone.
    _warn = []
    if table_od_qa["Identity gap"].max() >= 1e-9:
        _warn.append(f"total overdoses != fatal + nonfatal "
                     f"(max gap {table_od_qa['Identity gap'].max():.2e})")
    if abs(table_od_qa["Deaths/person"] - 1.0).max() >= 1e-9:
        _warn.append(f"deaths per person != 1 (max deviation "
                     f"{abs(table_od_qa['Deaths/person'] - 1.0).max():.2e})")
    for arm in ARMS:
        for hlab in v_housing:
            got = fatal_fraction(df_psa, arm, hlab)
            if np.isfinite(got) and not (-0.02 < got - _intended[hlab] < 0.01):
                _warn.append(f"{arm}/{hlab}: case fatality {got:.4f} vs intended "
                             f"{_intended[hlab]:.4f}")
    if _warn:
        print(f"\nOverdose validation warnings ({len(_warn)}); may be Monte Carlo "
              f"noise in small runs:")
        for _w in _warn:
            print(f"  - {_w}")
    else:
        print("Overdose identity and case-fatality checks passed.")

### 15.1 Main results table

Lifetime, discounted outcomes per person (fatal overdoses: expected number in a cohort of `n_i`), exported as `Table1_main_results.csv`.

In [ ]:
if analysis_gate("main results table"):
    # (column label, stem, scale, format)
    MAIN_TABLE_COLS = [
        ("Time spent in permanent housing (yrs)",           "years_H",      1.0, "{:.1f}"),
        ("Expected lifetime fatal overdoses in the cohort", "od_fatal",     n_i, "{:,.0f}"),
        ("Mean age at death (yrs)",                         "age_death",    1.0, "{:.1f}"),
        ("QALYs",                                           "qaly",         1.0, "{:.2f}"),
        ("Health care cost ($)",                            "health_cost",  1.0, "{:,.0f}"),
        ("Housing cost ($)",                                "housing_cost", 1.0, "{:,.0f}"),
        ("Total cost ($)",                                  "cost",         1.0, "{:,.0f}"),
    ]
    _sa_keys = [k for k in ["dsa_housing_cost", "dsa_qol", "dsa_placement"]
                if k in SCENARIO_BY_KEY]
    TABLE_SECTIONS = [("Base case analysis", ["base_case"])]
    if _sa_keys:
        TABLE_SECTIONS.append(("Sensitivity analyses", _sa_keys))
    SCENARIO_ROW_LABEL = {
        "base_case":        "",
        "dsa_housing_cost": "Doubled permanent housing cost\u1d43",
        "dsa_qol":          "Higher unhoused quality of life\u1d47",
        "dsa_placement":    "Higher chance of housing placement\u1d9c",
    }
    ARM_ROW_LABEL = {"comparator": "Status quo", "intervention": "Housing intervention"}

    _disp, _num = [], []
    for section, keys in TABLE_SECTIONS:
        _disp.append({"Scenario": section, "Arm": "",
                      **{lbl: "" for lbl, *_ in MAIN_TABLE_COLS}})
        for key in keys:
            scn = SCENARIO_BY_KEY[key]
            for ri, role in enumerate(["comparator", "intervention"]):
                cols = scenario_columns(scn, role)
                row = {"Scenario": SCENARIO_ROW_LABEL[key] if ri == 0 else "",
                       "Arm": ARM_ROW_LABEL[role]}
                nrow = {"Scenario": scn["label"], "Arm": ARM_ROW_LABEL[role],
                        "Arm code": scn[role]}
                for lbl, stem, scale, fmt in MAIN_TABLE_COLS:
                    m, lo, hi = ui(df_psa[cols[stem]], scale)
                    row[lbl] = f"{fmt.format(m)} [{fmt.format(lo)}\u2013{fmt.format(hi)}]"
                    nrow.update({f"{lbl} | mean": m, f"{lbl} | lo": lo,
                                 f"{lbl} | hi": hi})
                _disp.append(row)
                _num.append(nrow)
    table_main_results = pd.DataFrame(_disp)
    table_main_results_numeric = pd.DataFrame(_num)

    TABLE_FOOTNOTES = [
        f"Lifetime, per person, discounted at {annual_d_c:.0%} annually. "
        f"Costs in {COST_YEAR} USD.",
        f"Fatal overdoses are the expected number in a cohort of {n_i:,} "
        f"people; all other outcomes are per person.",
        f"Values are PSA means with 95% uncertainty intervals in brackets "
        f"({len(df_psa):,} parameter draws x {n_reps_psa} replications).",
        f"a  Annual cost of permanent housing doubled to "
        f"${c_housing_H * DSA1_HOUSING_MULT:,.0f}.",
        f"b  Overall utility for people experiencing homelessness raised from "
        f"{Q_PEH_BASE} to {Q_PEH_DSA2}.",
    ] + ([f"c  Intervention effect on 1-year housing access scaled "
          f"{DSA3_EFFECT_MULTIPLIER}x relative to Measure A."]
         if "dsa_placement" in SCENARIO_BY_KEY else []) + [
        "Only the base case and the threshold grid are simulated. The two cost "
        "and utility sensitivity analyses re-weight the same simulated "
        "life-courses, so they share the base case's non-monetary outcomes by "
        "construction.",
    ]

    print(f"Main results: lifetime outcomes, discounted, {COST_YEAR} USD")
    print("=" * 165)
    print(table_main_results.to_string(index=False))
    print("=" * 165)
    for _f in TABLE_FOOTNOTES:
        print(_f)

### 15.2 Internal validation of the housing calibration

For every draw and arm, the 1-year probabilities of ever reaching S and H among those who start unsheltered are compared at four levels: the calibration target; the continuous-time calibrated rates (should match to solver precision); the same rates stepped through the monthly chain without mortality (the gap is time discretisation); and the full simulation, where deaths count as non-achievers (the further gap is competing mortality plus Monte Carlo noise).

In [ ]:
if analysis_gate("housing validation"):
    def _nan_ui(x):
        x = np.asarray(x, dtype=float)
        return np.nanmean(x), np.nanpercentile(x, 2.5), np.nanpercentile(x, 97.5)

    # ------------------------------------------------------------
    # Per-draw comparison
    # ------------------------------------------------------------
    _rows = []
    for _, row in df_psa.iterrows():
        for arm in ARMS:
            started_U = row[f"{arm}_acc_started_U"]
            alive_U = row[f"{arm}_acc_alive_12mo"]
            rec = {"draw_id": int(row["draw_id"]), "arm": arm,
                   "surv_frac": alive_U / started_U if started_U > 0 else np.nan}
            for st in "SH":
                tgt = row[f"val_{arm}_target_{st}"]
                ct = row[f"val_{arm}_ct_{st}"]
                mo = row[f"val_{arm}_monthly_{st}"]
                sim = (row[f"{arm}_acc_ever_{st}"] / started_U
                       if started_U > 0 else np.nan)
                surv = (row[f"{arm}_acc_ever_{st}_alive"] / alive_U
                        if alive_U > 0 else np.nan)
                rec.update({
                    f"target_{st}": tgt, f"ct_{st}": ct, f"monthly_{st}": mo,
                    f"sim_{st}": sim, f"sim_survivors_{st}": surv,
                    f"ct_minus_target_{st}": ct - tgt,
                    f"monthly_minus_target_{st}": mo - tgt,
                    f"sim_minus_monthly_{st}": sim - mo,
                    f"sim_minus_target_{st}": sim - tgt,
                    f"monthly_rel_target_{st}": mo / tgt - 1,
                    f"sim_rel_monthly_{st}": sim / mo - 1,
                    f"sim_rel_target_{st}": sim / tgt - 1,
                })
            _rows.append(rec)
    df_housing_validation = pd.DataFrame(_rows)

    max_ct_error = max(df_housing_validation["ct_minus_target_S"].abs().max(),
                       df_housing_validation["ct_minus_target_H"].abs().max())
    print(f"Continuous-time calibration: largest error across {len(df_psa):,} draws "
          f"x {len(ARMS)} arms = {max_ct_error:.3e} "
          + ("(PASS)" if max_ct_error < 1e-8 else "(WARNING: targets not reproduced)"))

    # ------------------------------------------------------------
    # Summary by arm and outcome
    # ------------------------------------------------------------
    _summary = []
    for arm in ARMS:
        d = df_housing_validation[df_housing_validation["arm"] == arm]
        for st, outcome in [("S", "Ever reached sheltered housing"),
                            ("H", "Ever reached permanent housing")]:
            t_m, t_lo, t_hi = _nan_ui(d[f"target_{st}"])
            m_m, m_lo, m_hi = _nan_ui(d[f"monthly_{st}"])
            s_m, s_lo, s_hi = _nan_ui(d[f"sim_{st}"])
            v_m, _, _ = _nan_ui(d[f"sim_survivors_{st}"])
            _summary.append({
                "Arm": ARM_LABELS[arm], "Outcome": outcome,
                "Target mean": t_m, "Target 2.5%": t_lo, "Target 97.5%": t_hi,
                "Monthly mean": m_m, "Monthly 2.5%": m_lo, "Monthly 97.5%": m_hi,
                "Simulated mean": s_m, "Simulated 2.5%": s_lo,
                "Simulated 97.5%": s_hi, "Simulated survivors mean": v_m,
                "Monthly - target (pp)": 100 * d[f"monthly_minus_target_{st}"].mean(),
                "Simulation - monthly (pp)": 100 * d[f"sim_minus_monthly_{st}"].mean(),
                "Simulation - target (pp)": 100 * d[f"sim_minus_target_{st}"].mean(),
                "Monthly vs target (%)": 100 * d[f"monthly_rel_target_{st}"].mean(),
                "Simulation vs monthly (%)": 100 * d[f"sim_rel_monthly_{st}"].mean(),
                "Simulation vs target (%)": 100 * d[f"sim_rel_target_{st}"].mean(),
                "12-mo survival": d["surv_frac"].mean(),
            })
    table_housing_validation = pd.DataFrame(_summary)

    _disp = table_housing_validation.copy()
    for _lbl, _stem in [("Target (95% UI)", "Target"),
                        ("Exact monthly (95% UI)", "Monthly"),
                        ("Full simulation (95% UI)", "Simulated")]:
        _disp[_lbl] = [f"{r[f'{_stem} mean']:.4f} "
                       f"({r[f'{_stem} 2.5%']:.4f}\u2013{r[f'{_stem} 97.5%']:.4f})"
                       for _, r in table_housing_validation.iterrows()]
    _pp = ["Monthly - target (pp)", "Simulation - monthly (pp)", "Simulation - target (pp)"]

    print(f"\nHousing internal validation ({HOUSING_VALID_MONTHS}-month window; "
          f"denominator = everyone starting unsheltered, "
          f"{table_housing_validation['12-mo survival'].mean():.1%} survive)")
    print("=" * 145)
    print(_disp[["Arm", "Outcome", "Target (95% UI)", "Exact monthly (95% UI)",
                 "Full simulation (95% UI)"] + _pp].to_string(
        index=False, formatters={c: "{:+.3f}".format for c in _pp}))
    print("=" * 145)

## 16. Convergence diagnostics

Panels A–C show second-order convergence: running means of the base-case incremental cost, QALYs and INMB as PSA draws accumulate, with 95% Monte Carlo bands. Panel D shows first-order convergence: for each draw, how far the estimate from the first *n* saved replications lies from the estimate using all of them (root-mean-square relative deviation across draws).

In [ ]:
if analysis_gate("convergence"):
    CONV_WTP = 100_000
    CONV_SCN = SCENARIO_BY_KEY["base_case"]
    _c = scenario_columns(CONV_SCN, "comparator")
    _i = scenario_columns(CONV_SCN, "intervention")
    d_cost = (df_psa[_i["cost"]] - df_psa[_c["cost"]]).to_numpy()
    d_qaly = (df_psa[_i["qaly"]] - df_psa[_c["qaly"]]).to_numpy()
    inmb = CONV_WTP * d_qaly - d_cost
    n_draws = np.arange(1, len(d_cost) + 1)

    def _running(x):
        """Running mean and running standard error of the mean."""
        x = np.asarray(x, float)
        n = np.arange(1, x.size + 1)
        mean = np.cumsum(x) / n
        var = np.cumsum(x ** 2) / n - mean ** 2
        with np.errstate(invalid="ignore", divide="ignore"):
            sem = np.sqrt(np.maximum(var, 0) * n / np.maximum(n - 1, 1)) / np.sqrt(n)
        return mean, sem

    PANEL_SPEC = [
        ("Incremental cost", d_cost, "#D1751A", money),
        ("Incremental QALYs", d_qaly, "#0F8B8D", lambda v: f"{v:.4f}"),
        (f"INMB at ${CONV_WTP // 1000:,}K/QALY", inmb, "#1F6FB2", money),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 8.6), dpi=FIGDPI)

    # ---- Panels A-C: running means across draws ----
    for ax, (lbl, x, col, fmt) in zip(axes.ravel(), PANEL_SPEC):
        mean, sem = _running(x)
        ax.fill_between(n_draws, mean - 1.96 * sem, mean + 1.96 * sem,
                        color=col, alpha=0.20, linewidth=0, label="95% Monte Carlo band")
        ax.plot(n_draws, mean, lw=1.6, color=col, label="Running mean")
        ax.axhline(mean[-1], color="#D62728", ls="--", lw=1.2,
                   label=f"Final: {fmt(mean[-1])}")
        ax.set_xscale("log")
        ax.set_xlabel("Number of PSA draws", fontsize=10, fontweight="bold")
        ax.set_ylabel(lbl, fontsize=10, fontweight="bold")
        ax.set_title(f"{lbl} vs PSA draws", fontsize=11, fontweight="bold")
        ax.legend(fontsize=8, loc="best", framealpha=0.92)
        ax.grid(alpha=0.20, which="both")
        # Mark the draw from which the running mean stays within 1% of its final value
        tol = 0.01 * abs(mean[-1])
        if tol > 0:
            outside = np.where(np.abs(mean - mean[-1]) > tol)[0]
            k = int(outside[-1]) + 2 if outside.size else 1
            if k <= len(mean):
                ax.axvline(k, color="gray", ls=":", lw=1.0)
                ax.annotate(f"within 1% from\ndraw {k:,}", xy=(k, mean[-1]),
                            xytext=(6, -28), textcoords="offset points",
                            fontsize=7.5, color="#4D4D4D", linespacing=1.25)

    # ---- Panel D: empirical first-order convergence ----
    ax = axes[1, 1]
    _arm = CONV_SCN["intervention"]
    table_replication_convergence = None
    if df_reps is not None and len(df_reps) and _arm in set(df_reps["arm"]):
        emp = {}
        for lbl, outcome, col in [("Total cost", "cost_base", "#D1751A"),
                                  ("QALYs", "qaly_base", "#0F8B8D")]:
            if outcome not in df_reps.columns:
                continue
            M = (df_reps[df_reps["arm"] == _arm]
                 .pivot_table(index="draw_id", columns="rep_id", values=outcome)
                 .to_numpy())                           # draws x replications
            N = M.shape[1]
            ns = np.arange(1, N + 1)
            run = np.cumsum(M, axis=1) / ns
            # Parameter draw held fixed within each row, isolating first-order noise
            dev = np.sqrt(((run - run[:, [-1]]) ** 2).mean(axis=0))
            rel = dev / np.abs(run[:, -1]).mean()
            ax.plot(ns[:-1], rel[:-1], lw=1.9, color=col, marker="o", ms=3.2,
                    label=f"{lbl}, observed")
            ref = np.sqrt(np.maximum(1 / ns - 1 / N, 0))   # independent replications
            ref = ref / ref[0] * rel[0]
            ax.plot(ns[:-1], ref[:-1], color=col, ls=":", lw=1.1, alpha=0.75,
                    label=r"$\sqrt{1/n - 1/N}$")
            emp[lbl] = (ns, rel)

        ax.axvline(n_reps_psa, color="#D62728", ls="--", lw=1.3,
                   label=f"PSA uses {n_reps_psa} reps")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("Replications accumulated within a draw", fontsize=10,
                      fontweight="bold")
        ax.set_ylabel("Relative deviation from the full-replication estimate",
                      fontsize=9.5, fontweight="bold")
        ax.set_title("First-order convergence (empirical)", fontsize=11,
                     fontweight="bold")
        ax.legend(fontsize=7.5, framealpha=0.92)
        ax.grid(alpha=0.20, which="both")
        table_replication_convergence = pd.DataFrame({
            "replications": emp[next(iter(emp))][0],
            **{f"{k} relative deviation": v[1] for k, v in emp.items()}})
    else:
        ax.text(0.5, 0.5, "Replication-level outcomes not found\n"
                          "(set SAVE_PER_REPLICATION = True)",
                transform=ax.transAxes, ha="center", va="center", fontsize=10)
        ax.set_axis_off()

    fig.suptitle("PSA Convergence Diagnostics, Base Case", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(figpath("Figure7_convergence.png"), dpi=FIGDPI, bbox_inches="tight")
    plt.show()

    # ------------------------------------------------------------
    # Numeric summary
    # ------------------------------------------------------------
    _rows = []
    for lbl, x, _col, _f in PANEL_SPEC:
        mean, sem = _running(x)
        _rows.append({"Quantity": lbl, "Final mean": mean[-1], "MCSE": sem[-1],
                      "MCSE / |mean|": sem[-1] / abs(mean[-1]),
                      "95% MC half-width": 1.96 * sem[-1]})
    table_convergence = pd.DataFrame(_rows)

    print(f"Monte Carlo precision after {len(d_cost):,} PSA draws "
          f"(MCSE reflects simulation precision, not parameter uncertainty)")
    print(table_convergence.to_string(index=False, formatters={
        "Final mean": "{:,.4f}".format, "MCSE": "{:,.4f}".format,
        "MCSE / |mean|": "{:.4%}".format, "95% MC half-width": "{:,.4f}".format}))

    if table_replication_convergence is not None:
        _t = table_replication_convergence
        _half = _t.iloc[min(len(_t) // 2, len(_t) - 1)]
        print(f"\nFirst-order relative deviation with {int(_half['replications'])} of "
              f"{len(_t)} replications: "
              + ", ".join(f"{c.replace(' relative deviation', '')} {_half[c]:.3%}"
                          for c in _t.columns[1:]))

## 17. Threshold analysis

Incremental NMB and ICER of each threshold arm versus the status quo, at base-case and doubled permanent housing cost; the doubled-cost line re-costs the same simulated life-courses. The threshold M is where mean INMB crosses zero; when there is no crossing, the threshold lies outside the grid.

In [ ]:
if analysis_gate("threshold analysis"):
    THR_WTP = 100_000   # willingness to pay for the INMB panel
    THR_QALY = "base"   # utility variant for both lines
    THR_LINES = [
        ("base", "Base case", "#1F6FB2"),
        ("dsa1", f"Doubled permanent housing cost "
                 f"(${c_housing_H * DSA1_HOUSING_MULT:,.0f}/yr)", "#D1751A"),
    ]

    _rows = []
    for cost_var, cost_lbl, _col in THR_LINES:
        for arm in ARMS[1:]:
            dc = (df_psa[f"{arm}_cost_{cost_var}"] - df_psa[f"sq_cost_{cost_var}"]).to_numpy()
            dq = (df_psa[f"{arm}_qaly_{THR_QALY}"] - df_psa[f"sq_qaly_{THR_QALY}"]).to_numpy()
            inmb = THR_WTP * dq - dc
            _rows.append({
                "Cost scenario": cost_lbl, "cost_var": cost_var,
                "Arm": arm, "M": ARM_EFFECT[arm],
                "1-yr interim target": ARM_TARGETS[arm][0],
                "1-yr permanent target": ARM_TARGETS[arm][1],
                "Years housed": df_psa[f"{arm}_years_H"].mean(),
                "dCost": dc.mean(), "dQALY": dq.mean(),
                "ICER": dc.mean() / dq.mean() if dq.mean() != 0 else np.nan,
                "INMB": inmb.mean(),
                "INMB 2.5%": np.percentile(inmb, 2.5),
                "INMB 97.5%": np.percentile(inmb, 97.5),
                "P(CE)": (inmb >= 0).mean(),
            })
    table_threshold = (pd.DataFrame(_rows)
                       .sort_values(["cost_var", "M"]).reset_index(drop=True))

    # Threshold M: linear interpolation where mean INMB first changes sign
    M_THRESHOLD = {}
    for cost_var, _lbl, _col in THR_LINES:
        d = table_threshold[table_threshold["cost_var"] == cost_var]
        _M, _I = d["M"].to_numpy(), d["INMB"].to_numpy()
        _s = np.where(np.diff(np.sign(_I)))[0]
        M_THRESHOLD[cost_var] = (
            _M[_s[0]] + (0 - _I[_s[0]]) * (_M[_s[0] + 1] - _M[_s[0]])
            / (_I[_s[0] + 1] - _I[_s[0]]) if _s.size else np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 5.8), dpi=FIGDPI)

    # ---- Panel A: INMB ----
    ax = axes[0]
    ax.axhline(0, color="black", lw=1.3, alpha=0.85, zorder=4)
    for cost_var, cost_lbl, col in THR_LINES:
        d = table_threshold[table_threshold["cost_var"] == cost_var]
        ax.fill_between(d["M"], d["INMB 2.5%"], d["INMB 97.5%"],
                        color=col, alpha=0.15, linewidth=0, zorder=2)
        ax.plot(d["M"], d["INMB"], lw=2.6, color=col, marker="o", ms=6,
                mec="white", mew=1.0, zorder=3, label=cost_lbl)
        mt = M_THRESHOLD[cost_var]
        if np.isfinite(mt):
            ax.plot(mt, 0, marker="D", ms=9, color=col, mec="white", mew=1.1, zorder=6)
            ax.annotate(f"M = {mt:.2f}", xy=(mt, 0), xytext=(4, 10),
                        textcoords="offset points", fontsize=8.5, color=col,
                        fontweight="bold")
    ax.axvline(THRESHOLD_MEASURE_A, color="#4D4D4D", ls="--", lw=1.1, alpha=0.7)
    ax.annotate("Measure A\nas observed", xy=(THRESHOLD_MEASURE_A, 0),
                xytext=(5, 24), textcoords="offset points", fontsize=8,
                color="#4D4D4D", fontweight="bold", linespacing=1.25)
    ax.set_xlabel("Intervention effectiveness multiplier, M", fontsize=11,
                  fontweight="bold")
    ax.set_ylabel(f"Incremental NMB per person ({COST_YEAR} USD)", fontsize=11,
                  fontweight="bold")
    ax.set_title(f"A. Incremental net monetary benefit at "
                 f"${THR_WTP // 1000:,}K/QALY", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(fontsize=9, loc="best", framealpha=0.92,
              title="Cost scenario", title_fontsize=9)
    ax.grid(alpha=0.20)

    # ---- Panel B: ICER ----
    ax = axes[1]
    for cost_var, cost_lbl, col in THR_LINES:
        d = table_threshold[table_threshold["cost_var"] == cost_var]
        ax.plot(d["M"], d["ICER"], lw=2.5, color=col, marker="o", ms=6,
                mec="white", mew=1.0, label=cost_lbl)
    for lam, ls in zip(WTP_REPORT, [":", "--", "-."]):
        ax.axhline(lam, color="#4D4D4D", ls=ls, lw=1.1, alpha=0.8,
                   label=f"WTP ${lam // 1000:,}K/QALY")
    ax.axvline(THRESHOLD_MEASURE_A, color="#4D4D4D", ls="--", lw=1.1, alpha=0.5)
    ax.set_xlabel("Intervention effectiveness multiplier, M", fontsize=11,
                  fontweight="bold")
    ax.set_ylabel(f"ICER ({COST_YEAR} USD per QALY gained)", fontsize=11,
                  fontweight="bold")
    ax.set_title("B. Incremental cost-effectiveness ratio", fontsize=12,
                 fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(fontsize=8.5, loc="best", framealpha=0.92)
    ax.grid(alpha=0.20)

    fig.suptitle("Threshold Analysis: Intervention Effectiveness",
                 fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(figpath("Figure8_threshold_analysis.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

    _disp = table_threshold.copy()
    _disp["INMB (95% UI)"] = [
        f"{money(r['INMB'])} [{money(r['INMB 2.5%'])}\u2013{money(r['INMB 97.5%'])}]"
        for _, r in table_threshold.iterrows()]
    print(f"Threshold analysis at ${THR_WTP // 1000:,}K/QALY ({len(df_psa):,} PSA draws)")
    print("=" * 140)
    print(_disp[["Cost scenario", "M", "1-yr interim target", "1-yr permanent target",
                 "Years housed", "dCost", "dQALY", "ICER", "INMB (95% UI)",
                 "P(CE)"]].to_string(index=False, formatters={
        "M": "{:.2f}".format,
        "1-yr interim target": "{:.4f}".format,
        "1-yr permanent target": "{:.4f}".format,
        "Years housed": "{:.2f}".format, "dCost": "{:,.0f}".format,
        "dQALY": "{:.3f}".format, "ICER": "{:,.0f}".format,
        "P(CE)": "{:.1%}".format}))
    print("=" * 140)
    for cost_var, cost_lbl, _col in THR_LINES:
        mt = M_THRESHOLD[cost_var]
        if np.isfinite(mt):
            print(f"{cost_lbl}: threshold at M = {mt:.3f} ({mt:.1%} of Measure A's "
                  f"observed housing-access gain).")
        else:
            d = table_threshold[table_threshold["cost_var"] == cost_var]
            side = "above" if d["INMB"].iloc[0] > 0 else "below"
            print(f"{cost_lbl}: mean INMB is {side} zero at every grid point; the "
                  f"threshold lies outside M in [{d['M'].min():.2f}, {d['M'].max():.2f}].")

## 18. ENMB, EVPI and the acceptability frontier

ENMB(λ) is the mean incremental NMB across draws. EVPI(λ) = E[max over strategies of NMB] − max over strategies of E[NMB], the expected value of resolving all parameter uncertainty; population EVPI is illustrative (120,000 people over 10 years). The cost-effectiveness acceptability frontier (CEAF) follows the strategy that is optimal in expectation and reports its probability of being cost-effective, so it can fall below 0.5 when the NMB distribution is skewed.

In [ ]:
if analysis_gate("ENMB and EVPI"):
    N_PEH_OUD_POP = 120_000   # illustrative population for population EVPI
    T_HORIZON     = 10        # decision horizon, years

    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        c_cmp = df_psa[S["cmp_cols"]["cost"]].to_numpy()
        c_int = df_psa[S["int_cols"]["cost"]].to_numpy()
        q_cmp = df_psa[S["cmp_cols"]["qaly"]].to_numpy()
        q_int = df_psa[S["int_cols"]["qaly"]].to_numpy()

        enmb = np.zeros(WTP_GRID.size)
        evpi = np.zeros(WTP_GRID.size)
        for k, lam in enumerate(WTP_GRID):
            nmb_cmp = lam * q_cmp - c_cmp
            nmb_int = lam * q_int - c_int
            enmb[k] = (nmb_int - nmb_cmp).mean()
            evpi[k] = (np.maximum(nmb_cmp, nmb_int).mean()
                       - max(nmb_cmp.mean(), nmb_int.mean()))
        S["enmb"] = enmb
        S["evpi"] = evpi
        S["pop_evpi"] = evpi * N_PEH_OUD_POP * T_HORIZON
        S["ceaf"] = np.where(enmb >= 0, S["ceac"], 1.0 - S["ceac"])
        S["frontier_is_intervention"] = enmb >= 0

        # Break-even WTP: linear interpolation where ENMB first changes sign
        sign_change = np.where(np.diff(np.sign(enmb)))[0]
        if sign_change.size:
            j = sign_change[0]
            x1, x2 = WTP_GRID[j], WTP_GRID[j + 1]
            y1, y2 = enmb[j], enmb[j + 1]
            S["breakeven_wtp"] = x1 + (0 - y1) * (x2 - x1) / (y2 - y1)
        else:
            S["breakeven_wtp"] = np.nan

    df_wtp_curves = pd.DataFrame({"WTP": WTP_GRID})
    for scn in SCENARIOS:
        k, S = scn["key"], psa_scenario[scn["key"]]
        for curve in ["ceac", "enmb", "evpi", "pop_evpi"]:
            df_wtp_curves[f"{k}_{curve}"] = S[curve]
    for scn in SCENARIOS:
        df_wtp_curves[f"{scn['key']}_ceaf"] = psa_scenario[scn["key"]]["ceaf"]

    # ------------------------------------------------------------
    # Summary at the reported thresholds
    # ------------------------------------------------------------
    _j_report = [int(np.argmin(np.abs(WTP_GRID - lam))) for lam in WTP_REPORT]
    _rows = []
    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        row = {"Scenario": scn["label"],
               "Break-even WTP ($/QALY)": ("--" if np.isnan(S["breakeven_wtp"])
                                           else f"${S['breakeven_wtp']:,.0f}")}
        for lam, j in zip(WTP_REPORT, _j_report):
            row[f"ENMB at ${lam // 1000}k"] = money(S["enmb"][j])
            row[f"EVPI at ${lam // 1000}k"] = money(S["evpi"][j])
        peak = int(np.argmax(S["evpi"]))
        row["Peak EVPI"] = money(S["evpi"][peak])
        row["Peak EVPI at WTP"] = f"${WTP_GRID[peak]:,}"
        _rows.append(row)
    table_enmb_evpi = pd.DataFrame(_rows)

    table_ceac_report = pd.DataFrame({"WTP ($/QALY)": [f"${l:,}" for l in WTP_REPORT]})
    for scn in SCENARIOS:
        table_ceac_report[scn["label"]] = [
            f"{psa_scenario[scn['key']]['ceac'][j] * 100:.1f}%" for j in _j_report]

    print("ENMB and EVPI per person, by scenario")
    print("=" * 140)
    print(table_enmb_evpi.to_string(index=False))
    print("=" * 140)
    print(f"Population EVPI (N = {N_PEH_OUD_POP:,}, {T_HORIZON} years) is in "
          f"df_wtp_curves / WTP_curves.csv.")
    print("\nProbability the intervention is cost-effective")
    print(table_ceac_report.to_string(index=False))

## 19. Figures

The first cell defines the shared palette and plotting helpers.

In [ ]:
if analysis_gate("figure palette"):
    # Colour-blind-safe scenario colours; red is reserved for means and break-even marks
    SCN_COLORS = {
        "base_case":        "#1F6FB2",   # blue
        "dsa_housing_cost": "#D1751A",   # orange
        "dsa_qol":          "#0F8B8D",   # teal
        "dsa_placement":    "#7B4EA3",   # purple
    }
    MEAN_COLOR   = "#D62728"
    THRESH_COLOR = "#4D4D4D"
    WTP_COLORS   = {50_000: "#4C9BD1", 100_000: "#D1751A",
                    150_000: "#0F8B8D", 200_000: "#7B4EA3"}
    WTP_LINES    = [50_000, 100_000, 150_000, 200_000]   # thresholds marked on WTP axes
    CE_PLANE_WTP = [(50_000, ":"), (100_000, "--"), (200_000, "-.")]
    WTP_XLABEL   = "Willingness-to-pay threshold ($1,000 per QALY gained)"
    AX_LABEL_KW  = dict(fontsize=11, fontweight="bold")
    TITLE_KW     = dict(fontsize=13, fontweight="bold")

    def _scn_color(key):
        return SCN_COLORS.get(key, "#808080")

    def _wtp_color(lam):
        return WTP_COLORS.get(lam, "#808080")

    def _money_axis(ax, which="y"):
        fmt = mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
        (ax.yaxis if which == "y" else ax.xaxis).set_major_formatter(fmt)

    def _draw_wtp_lines(ax, top_labels=True, y_top=1.015):
        """Vertical dashed lines at each WTP threshold, optionally labelled."""
        for lam in WTP_LINES:
            ax.axvline(lam / 1000, color=THRESH_COLOR, ls="--", lw=0.9, alpha=0.55,
                       zorder=1)
            if top_labels:
                ax.text(lam / 1000, y_top, f"${lam // 1000}K", ha="center",
                        va="bottom", fontsize=8, color=THRESH_COLOR,
                        fontweight="bold", transform=ax.get_xaxis_transform())

    def _hide_unused_panels(axes, n_used, xlabel=None):
        """Hide empty small-multiple panels. With a shared x-axis, the panel
        above each hidden one gets the tick labels and `xlabel`."""
        flat, ncols = axes.ravel(), axes.shape[1]
        for k in range(n_used, flat.size):
            flat[k].set_visible(False)
            if xlabel and k >= ncols:
                flat[k - ncols].xaxis.set_tick_params(labelbottom=True)
                flat[k - ncols].set_xlabel(xlabel, fontsize=10, fontweight="bold")

### Figure 1. Cost-effectiveness plane

In [ ]:
if analysis_gate("Figure 1"):
    fig, axes = plt.subplots(2, 2, figsize=(13, 10), dpi=FIGDPI)
    for ax, scn in zip(axes.ravel(), SCENARIOS):
        S = psa_scenario[scn["key"]]
        dq, dc = S["delta_qaly"], S["delta_cost"]

        ax.scatter(dq, dc, s=11, alpha=0.30, color=_scn_color(scn["key"]),
                   edgecolors="none", zorder=2, label="PSA samples")
        x_lo, x_hi = min(dq.min(), 0.0), max(dq.max(), 1e-3)
        pad = 0.04 * (x_hi - x_lo)
        xs = np.linspace(x_lo - pad, x_hi + pad, 100)
        for lam, ls in CE_PLANE_WTP:
            ax.plot(xs, lam * xs, ls=ls, lw=1.3, color=THRESH_COLOR, alpha=0.85,
                    zorder=3, label=f"WTP ${lam:,}/QALY gained")
        ax.scatter(dq.mean(), dc.mean(), s=48, marker="D", color=MEAN_COLOR,
                   edgecolors="white", linewidths=0.8, zorder=6,
                   label=f"Mean (ICER {money(S['icer'])}/QALY gained)")

        ax.axhline(0, color="gray", lw=0.6, alpha=0.45, zorder=1)
        ax.axvline(0, color="gray", lw=0.6, alpha=0.45, zorder=1)
        ax.set_xlim(xs[0], xs[-1])
        ax.set_title(scn["label"], fontsize=11, fontweight="bold")
        ax.set_xlabel("Incremental QALYs gained", fontsize=10)
        ax.set_ylabel(f"Incremental cost ({COST_YEAR} USD)", fontsize=10)
        _money_axis(ax)
        ax.legend(fontsize=7.5, loc="upper left", framealpha=0.92)
        ax.grid(alpha=0.18)
    _hide_unused_panels(axes, len(SCENARIOS))

    fig.suptitle("Cost-Effectiveness Plane by Scenario", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(figpath("Figure1_CE_plane_by_scenario.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### Figure 2. Cost-effectiveness acceptability curves

Colour identifies the strategy. Solid segments indicate the strategy with the highest expected net monetary benefit; dashed segments show the other strategy. The switch point occurs where expected incremental net monetary benefit equals zero.

In [ ]:
if analysis_gate("Figure 2"):
    # Dash patterns in points: "--" scales with path length and looks uneven
    CEAC_DASH_LOWER = (3.0, 1.8)
    CEAC_DASH_CROSS = (3.5, 2.2)
    LW_UPPER, LW_LOWER = 2.3, 2.0

    def _lighten(color, amount=0.52):
        """Blend a colour toward white."""
        rgb = np.array(mpl.colors.to_rgb(color))
        return tuple((1 - amount) * rgb + amount * np.ones(3))

    fig, axes = plt.subplots(2, 2, figsize=(13, 9.4), dpi=FIGDPI,
                             sharex=True, sharey=True)
    _xing = []
    for ax, scn in zip(axes.ravel(), SCENARIOS):
        S = psa_scenario[scn["key"]]
        col = _scn_color(scn["key"])
        col_sq = _lighten(col)
        ceac_int = np.asarray(S["ceac"], float)
        ceac_sq = 1.0 - ceac_int
        x = WTP_GRID / 1000

        # Use the expected-NMB results from the earlier "ENMB and EVPI" block.
        int_optimal = np.asarray(S["frontier_is_intervention"], dtype=bool)

        # Show both CEACs dashed; make the expected-optimal strategy solid.
        for prob, color, optimal in [
            (ceac_sq, col_sq, ~int_optimal),
            (ceac_int, col, int_optimal),
        ]:
            ax.plot(x, prob, color=color, lw=LW_LOWER,
                    ls=(0, CEAC_DASH_LOWER))
            ax.plot(x, np.where(optimal, prob, np.nan),
                    color=color, lw=LW_UPPER, ls="-")

        # Switch where expected incremental NMB equals zero, not P(CE) = 0.5.
        x_cross = S["breakeven_wtp"]
        if np.isfinite(x_cross):
            xc = x_cross / 1000
            yc = np.interp(xc, x, ceac_int)
            ax.plot([xc, xc], [0, yc], color=col,
                    ls=(0, CEAC_DASH_CROSS), lw=1.2)
            ax.plot(xc, yc, marker="o", ms=7, color=col,
                    mec="white", mew=1.1)

        ax.axhline(0.5, color="gray", ls=":", lw=0.9, alpha=0.7, zorder=1)
        _draw_wtp_lines(ax)

        _handles = [
            plt.Line2D([], [], color=col_sq, lw=LW_UPPER, ls="-", label="Status quo"),
            plt.Line2D([], [], color=col, lw=LW_UPPER, ls="-",
                       label="Housing intervention"),
        ]
        if np.isfinite(x_cross):
            _handles.append(plt.Line2D([], [], color=col, lw=0, marker="o", ms=7,
                                       mec="white", mew=1.1, label="Switch point"))
        ax.legend(handles=_handles, fontsize=8.2, loc="lower right",
                  framealpha=0.92, borderpad=0.6, labelspacing=0.45,
                  handlelength=1.9)
        ax.set_title(scn["label"], fontsize=11, fontweight="bold", pad=18)
        ax.set_xlim(0, 300)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.18)
        _xing.append({"Scenario": scn["label"],
                      "Switch point (equal expected NMB)": x_cross,
                      "ICER (mean break-even)": S["icer"]})
    table_ceac_crossing = pd.DataFrame(_xing)

    for ax in axes[-1]:
        ax.set_xlabel(WTP_XLABEL, fontsize=10, fontweight="bold")
    for ax in axes[:, 0]:
        ax.set_ylabel("Probability cost-effective", fontsize=10, fontweight="bold")
    _hide_unused_panels(axes, len(SCENARIOS), WTP_XLABEL)

    fig.suptitle("Cost-Effectiveness Acceptability Frontier", fontsize=13,
                 fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.955])
    plt.savefig(figpath("Figure2_CEAC_by_scenario.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### Figure 3. Expected net monetary benefit

In [ ]:
if analysis_gate("Figure 3"):
    _all_enmb = np.concatenate([psa_scenario[s["key"]]["enmb"] for s in SCENARIOS])
    _span = max(_all_enmb.max() - _all_enmb.min(), 1.0)
    _Y0 = _all_enmb.min() - 0.08 * _span
    _Y1 = _all_enmb.max() + 0.10 * _span

    fig, axes = plt.subplots(2, 2, figsize=(13, 9.2), dpi=FIGDPI,
                             sharex=True, sharey=True)
    for ax, scn in zip(axes.ravel(), SCENARIOS):
        S = psa_scenario[scn["key"]]
        col = _scn_color(scn["key"])
        ax.fill_between(WTP_GRID / 1000, 0, S["enmb"], color=col, alpha=0.30,
                        linewidth=0, zorder=2)
        ax.plot(WTP_GRID / 1000, S["enmb"], lw=2.3, color=col, zorder=3)
        ax.axhline(0, color="black", lw=1.2, alpha=0.85, zorder=4)
        _draw_wtp_lines(ax)

        be = S["breakeven_wtp"]
        if np.isfinite(be):
            ax.plot(be / 1000, 0, marker="o", ms=6.5, color=MEAN_COLOR,
                    mec="white", mew=0.9, zorder=6)
            ax.annotate(f"Break-even\n${be:,.0f}/QALY", xy=(be / 1000, 0),
                        xytext=(8, -34), textcoords="offset points",
                        fontsize=7.8, color=MEAN_COLOR, fontweight="bold",
                        linespacing=1.25)

        ax.set_title(scn["label"], fontsize=11, fontweight="bold", pad=18)
        ax.set_xlim(0, 300)
        ax.set_ylim(_Y0, _Y1)
        _money_axis(ax)
        ax.grid(alpha=0.18)

    for ax in axes[-1]:
        ax.set_xlabel(WTP_XLABEL, fontsize=10, fontweight="bold")
    for ax in axes[:, 0]:
        ax.set_ylabel(f"Expected incremental NMB ({COST_YEAR} USD)", fontsize=10,
                      fontweight="bold")
    _hide_unused_panels(axes, len(SCENARIOS), WTP_XLABEL)

    fig.suptitle("Expected Net Monetary Benefit", fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(figpath("Figure3_ENMB_by_scenario.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### Figure 4. Expected value of perfect information

In [ ]:
if analysis_gate("Figure 4"):
    fig, ax = plt.subplots(figsize=(9.8, 5.8), dpi=FIGDPI)
    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        col = _scn_color(scn["key"])
        ax.fill_between(WTP_GRID / 1000, 0, S["evpi"], color=col, alpha=0.10,
                        linewidth=0, zorder=2)
        ax.plot(WTP_GRID / 1000, S["evpi"], lw=2.3, color=col, zorder=3,
                label=scn["label"])
        peak = int(np.argmax(S["evpi"]))
        ax.plot(WTP_GRID[peak] / 1000, S["evpi"][peak], marker="v", ms=6,
                color=col, mec="white", mew=0.7, zorder=5)

    _draw_wtp_lines(ax)
    ax.set_xlabel(WTP_XLABEL, **AX_LABEL_KW)
    ax.set_ylabel(f"EVPI per person ({COST_YEAR} USD)", **AX_LABEL_KW)
    ax.set_title("Expected Value of Perfect Information", pad=20, **TITLE_KW)
    _money_axis(ax)
    ax.set_xlim(0, 300)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=9, framealpha=0.92)
    ax.grid(alpha=0.20)
    plt.tight_layout()
    plt.savefig(figpath("Figure4_EVPI_by_scenario.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### Figure 5. Incremental NMB distribution, base case

In [ ]:
if analysis_gate("Figure 5"):
    S = psa_scenario["base_case"]
    fig, ax = plt.subplots(figsize=(9.8, 5.8), dpi=FIGDPI)
    for lam in WTP_REPORT:
        v = S["inmb"][lam]
        col = _wtp_color(lam)
        ax.hist(v, bins=40, alpha=0.42, color=col, edgecolor="white", linewidth=0.4,
                label=f"WTP ${lam // 1000}K/QALY gained  "
                      f"(P(CE) = {(v >= 0).mean() * 100:.0f}%)")
        ax.axvline(v.mean(), color=col, ls="--", lw=1.5, alpha=0.95)
    ax.axvline(0, color="black", lw=1.5, zorder=4)
    ax.set_xlabel(f"Incremental net monetary benefit ({COST_YEAR} USD)", **AX_LABEL_KW)
    ax.set_ylabel("Number of PSA samples", **AX_LABEL_KW)
    ax.set_title("Incremental NMB Distribution, Base Case", **TITLE_KW)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x / 1000:,.0f}K"))
    ax.legend(fontsize=9, framealpha=0.92)
    ax.grid(alpha=0.20)
    plt.tight_layout()
    plt.savefig(figpath("Figure5_INMB_distribution_base_case.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### Figure 6. Incremental cost and QALY distributions, base case

In [ ]:
if analysis_gate("Figure 6"):
    S = psa_scenario["base_case"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.9), dpi=FIGDPI)

    axes[0].hist(S["delta_cost"], bins=40, color=SCN_COLORS["dsa_housing_cost"],
                 edgecolor="white", linewidth=0.4, alpha=0.85)
    axes[0].axvline(S["delta_cost"].mean(), color=MEAN_COLOR, ls="--", lw=1.9,
                    label=f"Mean {money(S['delta_cost'].mean())}")
    axes[0].set_xlabel(f"Incremental cost ({COST_YEAR} USD)", fontsize=10)
    axes[0].set_title("Incremental Cost", fontsize=11, fontweight="bold")
    axes[0].xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x / 1000:,.0f}K"))

    axes[1].hist(S["delta_qaly"], bins=40, color=SCN_COLORS["dsa_qol"],
                 edgecolor="white", linewidth=0.4, alpha=0.85)
    axes[1].axvline(S["delta_qaly"].mean(), color=MEAN_COLOR, ls="--", lw=1.9,
                    label=f"Mean {S['delta_qaly'].mean():.3f}")
    axes[1].set_xlabel("Incremental QALYs gained", fontsize=10)
    axes[1].set_title("Incremental QALYs Gained", fontsize=11, fontweight="bold")

    for a in axes:
        a.axvline(0, color="black", lw=1.1, alpha=0.7)
        a.set_ylabel("Number of PSA samples", fontsize=10)
        a.legend(fontsize=9, framealpha=0.92)
        a.grid(alpha=0.20)

    fig.suptitle("Distribution of Incremental Outcomes, Base Case", **TITLE_KW)
    plt.tight_layout()
    plt.savefig(figpath("Figure6_incremental_distributions.png"), dpi=FIGDPI,
                bbox_inches="tight")
    plt.show()

### CEAC and CEAF at reported thresholds

In [ ]:
if analysis_gate("CEAC/CEAF summary"):
    _rows = []
    for scn in SCENARIOS:
        S = psa_scenario[scn["key"]]
        row = {"Scenario": scn["label"]}
        for lam in WTP_LINES:
            j = int(np.argmin(np.abs(WTP_GRID - lam)))
            row[f"CEAC ${lam // 1000}K"] = f"{S['ceac'][j] * 100:.1f}%"
            row[f"CEAF ${lam // 1000}K"] = f"{S['ceaf'][j] * 100:.1f}%"
            row[f"Optimal ${lam // 1000}K"] = ("Intervention"
                                               if S["frontier_is_intervention"][j]
                                               else "Status quo")
        _rows.append(row)
    table_ceac_ceaf = pd.DataFrame(_rows)

    print("CEAC (P(intervention cost-effective)) and CEAF (probability for the "
          "strategy optimal in expectation)")
    print("=" * 130)
    print(table_ceac_ceaf.to_string(index=False))
    print("=" * 130)

## 20. Export

Writes draw- and replication-level results, all tables, model matrices, the run configuration and a `MANIFEST.json` to the results directory. Shard jobs skip this step so that concurrently finishing jobs never write the same files.

In [ ]:
def _save_frame(df, stem, sub="replications"):
    """Save as parquet (pickle if parquet is unavailable) plus gzipped CSV."""
    p = rpath(sub, f"{stem}.parquet")
    try:
        df.to_parquet(p, index=False)
    except Exception:
        p = rpath(sub, f"{stem}.pkl")
        df.to_pickle(p)
    pc = rpath(sub, f"{stem}.csv.gz")
    df.to_csv(pc, index=False, compression="gzip")
    return p, pc


# (variable, file name, description)
EXPORT_TABLES = [
    ("table_main_results",            "Table1_main_results.csv",
     "main results, mean and 95% UI, manuscript layout"),
    ("table_main_results_numeric",    "Table1_main_results_numeric.csv",
     "main results, numeric means and interval bounds"),
    ("table_cost_conversion",         "Table0_cost_conversion.csv",
     "source cost values, deflators and PSA distributions"),
    ("table_psa_absolute_display",    "Table2_PSA_absolute_display.csv",
     "absolute outcomes by scenario and arm"),
    ("table_psa_absolute_numeric",    "Table2_PSA_absolute_numeric.csv",
     "absolute outcomes, numeric"),
    ("table_psa_incremental_display", "Table3_PSA_incremental_display.csv",
     "incremental outcomes, ICER, INMB, P(CE)"),
    ("table_psa_incremental_numeric", "Table3_PSA_incremental_numeric.csv",
     "incremental outcomes, numeric"),
    ("table_enmb_evpi",               "Table4_ENMB_EVPI.csv",
     "ENMB and EVPI at reported thresholds"),
    ("table_ceac_report",             "Table5_CEAC.csv",
     "probability cost-effective at reported thresholds"),
    ("table_ceac_crossing",           "Table6_CEAC_crossing.csv",
     "CEAC crossing and ICER by scenario"),
    ("table_threshold",               "Table7_threshold_analysis.csv",
     "threshold analysis over the housing-effect grid"),
    ("table_convergence",             "Table8_convergence.csv",
     "Monte Carlo precision after all draws"),
    ("table_replication_convergence", "Table8b_replication_convergence.csv",
     "empirical convergence over accumulated replications 1..n"),
    ("table_od_qa",                   "TableS2_overdose_QA.csv",
     "simulated overdose events and case fatality"),
    ("table_housing_validation",      "TableS3_housing_validation.csv",
     "PSA internal validation of 1-year housing access"),
    ("df_housing_validation",         "TableS3_housing_validation_by_draw.csv",
     "per-draw housing validation"),
    ("table_shards",                  "TableS4_shards.csv",
     "shard inventory for this run"),
    ("df_ceac",                       "CEAC_full_grid.csv",
     "CEAC across the full WTP grid"),
    ("df_wtp_curves",                 "WTP_curves.csv",
     "CEAC, CEAF, ENMB, EVPI and population EVPI across the WTP grid"),
]

EXPORT_MATRICES = [
    "health_cost_matrix", "qaly_matrix_base", "qaly_matrix_dsa2",
    "at_utility_base", "at_utility_dsa2", "total_od_base", "fatal_od_base",
    "nonfatal_od_base", "oud_nonod_base", "bg_mort_lut", "home_mult_lut",
    "baseline_cost_lut", "qaly_age_mult", "at_relapse_base", "OUD_trans_base",
    "init_age", "init_sex_int", "init_housing_int", "init_OUD_int",
]


def save_all():
    """Save results, tables, matrices and configuration; return the manifest."""
    g = globals()
    manifest = {"saved_at": datetime.datetime.now().isoformat(timespec="seconds"),
                "results_dir": RESULTS_DIR, "model_version": MODEL_VERSION,
                "run_fingerprint": RUN_FINGERPRINT, "quick_test": bool(QUICK_TEST),
                "analysis_complete": bool(RUN_ANALYSIS), "files": {}}

    def note(key, path, desc):
        manifest["files"][key] = {"path": os.path.relpath(path, RESULTS_DIR),
                                  "description": desc}

    # ---- Draw- and replication-level results ----
    p, pc = _save_frame(df_psa, "df_psa")
    note("df_psa", p, "PSA draw-level results and sampled parameters")
    note("df_psa_csv", pc, "same, gzipped CSV")
    if df_reps is not None and len(df_reps):
        p, pc = _save_frame(df_reps, "df_reps")
        note("df_reps", p, "replication-level outcomes (draw_id, arm, rep_id)")
        note("df_reps_csv", pc, "same, gzipped CSV")

    # ---- Tables ----
    for name, fname, desc in EXPORT_TABLES:
        obj = g.get(name)
        if isinstance(obj, pd.DataFrame) and len(obj):
            pth = rpath("tables", fname)
            obj.to_csv(pth, index=False)
            note(name, pth, desc)
    if "TABLE_FOOTNOTES" in g:
        pth = rpath("tables", "Table1_main_results_footnotes.txt")
        with open(pth, "w") as f:
            f.write("\n".join(TABLE_FOOTNOTES) + "\n")
        note("footnotes", pth, "footnotes for the main table")

    # ---- Model matrices and baseline cohort ----
    mats = {k: np.asarray(g[k]) for k in EXPORT_MATRICES if k in g}
    mats.update({f"housing_cost_matrix_{k}": np.asarray(v)
                 for k, v in housing_cost_mats_base.items()})
    mats.update({f"housing_trans_{k}": np.asarray(v) for k, v in ARM_TRANS.items()})
    pth = rpath("parameters", "matrices.npz")
    np.savez_compressed(pth, **mats)
    note("matrices", pth, "model matrices and the baseline cohort")

    # ---- Run configuration ----
    cfg = {
        "model_version": MODEL_VERSION, "run_fingerprint": RUN_FINGERPRINT,
        "quick_test": bool(QUICK_TEST), "cost_year": COST_YEAR,
        "n_i": n_i, "n_psa": int(len(df_psa)), "n_reps_psa": n_reps_psa,
        "psa_reps_per_chunk": PSA_REPS_PER_CHUNK,
        "save_per_replication": bool(SAVE_PER_REPLICATION),
        "rep_track": list(CHUNK_TRACK),
        "annual_d_c": annual_d_c, "annual_d_e": annual_d_e,
        "arms": ARMS, "arm_labels": ARM_LABELS, "arm_effect": ARM_EFFECT,
        "arm_targets": {k: list(v) for k, v in ARM_TARGETS.items()},
        "arm_rates": {k: list(v) for k, v in ARM_RATES.items()},
        "int_arm": INT_ARM, "enh_arm": ENH_ARM,
        "threshold_grid": THRESHOLD_GRID,
        "threshold_measure_a": THRESHOLD_MEASURE_A,
        "scenarios": SCENARIOS,
        "wtp_grid": [int(x) for x in WTP_GRID],
        "wtp_report": [int(x) for x in WTP_REPORT],
        "housing_valid_months": HOUSING_VALID_MONTHS,
        "v_housing": v_housing, "v_OUD": v_OUD,
        "c_housing_H": c_housing_H, "dsa1_housing_mult": DSA1_HOUSING_MULT,
        "q_peh_base": Q_PEH_BASE, "q_peh_dsa2": Q_PEH_DSA2,
        "deflators": {k: v[0] for k, v in DEFLATORS.items()},
        "costs": COSTS, "cost_psa": {k: list(v) for k, v in COST_PSA.items()},
        "od_rate_reported": OD_RATE_REPORTED,
        "od_survival_reported": OD_SURVIVAL_REPORTED,
        "p_fatal": P_FATAL_BY_HOUSING,
        "run_config": RUN_CONFIG,
    }
    pth = os.path.join(RESULTS_DIR, "run_config.json")
    with open(pth, "w") as f:
        json.dump(cfg, f, indent=2, default=_json_default)
    note("run_config", pth, "full run configuration")

    with open(os.path.join(RESULTS_DIR, "MANIFEST.json"), "w") as f:
        json.dump(manifest, f, indent=2, default=_json_default)

    print(f"Saved {len(manifest['files'])} objects to {RESULTS_DIR}")
    for k, v in manifest["files"].items():
        print(f"  {k:32s} -> {v['path']}")
    return manifest


if RUN_ANALYSIS:
    manifest = save_all()
    print("\nRun complete.")
else:
    manifest = None
    print("Export skipped: this job's PSA records are already saved in checkpoints/. "
          "Run the final merge job (PEH_OUD_SKIP_ANALYSIS=0) after all shards finish.")